# GraphSAGE and HGT Cold-OD Baselines v2

This notebook locks down the current Cold-OD graph results with a stricter and more reproducible protocol.

**Version 2 changes compared with the first graph notebook**

1. Segment summaries are computed from the **test split only**.
2. Paired graph-vs-baseline summaries are computed from the **test split only**.
3. GraphSAGE and HGT save three validation-selected checkpoints:
   - `best_val_pinball`
   - `best_val_q75`
   - `best_val_iqr`
4. The full graph experiment is repeated with five seeds:
   - `7, 42, 123, 2026, 535`

The target task is the FREIGHT-MNet Cold-OD setting: train on 2018--2022 rows from training FAF OD pairs, validate on 2023 rows from unseen validation OD pairs, and test on 2024 rows from unseen test OD pairs.  The models predict annual truck travel-time quantiles `truck_q25`, `truck_q50`, and `truck_q75`.

The graph models are intentionally still **baseline graph models**, not the final D-CQHGT.  They use FAF-zone nodes, training-only graph edges, a residualized monotone quantile head, and the same Cold-OD split as the previous non-graph baselines.

## 1. Environment setup and imports

This cell imports the packages used by the graph experiment.  The notebook deliberately avoids a subprocess preflight check because earlier environments mixed NumPy 1.x and NumPy 2.x compiled packages.  The active kernel itself is checked directly.

The graph section requires `torch` and `torch_geometric`.  If PyTorch Geometric is unavailable, the notebook stops early with a clear message instead of failing later during model construction.

In [1]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

# Disable optional pandas acceleration paths that can trigger binary-extension issues.
os.environ.setdefault("NUMEXPR_MAX_THREADS", "1")

import numpy as np
import pandas as pd

pd.set_option("compute.use_numexpr", False)
pd.set_option("compute.use_bottleneck", False)

import torch
from torch import nn
import torch.nn.functional as F

try:
    from torch_geometric.nn import HGTConv, SAGEConv
    TORCH_GEOMETRIC_AVAILABLE = True
    TORCH_GEOMETRIC_IMPORT_ERROR = None
except Exception as exc:  # pragma: no cover - only used when the local kernel lacks PyG.
    TORCH_GEOMETRIC_AVAILABLE = False
    TORCH_GEOMETRIC_IMPORT_ERROR = repr(exc)

print("Python executable:", os.sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
print("PyTorch Geometric available:", TORCH_GEOMETRIC_AVAILABLE)
if not TORCH_GEOMETRIC_AVAILABLE:
    print("PyG import error:", TORCH_GEOMETRIC_IMPORT_ERROR)
    raise RuntimeError("torch_geometric is required for GraphSAGE/HGT Cold-OD baselines.")

Python executable: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
NumPy: 2.4.5
Pandas: 2.3.3
Torch: 2.12.0+cu126
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4050 Laptop GPU
PyTorch Geometric available: True


## 2. Experiment configuration

All paths, split parameters, graph hyperparameters, checkpoint policy, and seeds are centralized here.  The default paths match the project layout used in the previous notebooks.

The notebook first tries to reuse the previous Cold-OD split from `predictions_cold_od_all_splits.parquet`.  This is important because graph models should be compared against the Cold-OD MLP/LightGBM baselines on exactly the same held-out OD pairs.

In [2]:
@dataclass
class ExperimentConfig:
    """Configuration for the multi-seed Cold-OD graph experiment."""

    # Project and data paths.
    data_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data")
    scope: str = "east_plus_gulf"
    run_name: str = "graphsage_hgt_cold_od_baselines_v2_notebook"

    supervised_relative_path: str = (
        r"08_processed\model_ready\freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet"
    )
    manifest_relative_path: str = (
        r"08_processed\model_ready\_metadata\freight_mnet_supervised_feature_manifest.json"
    )

    # Previous Cold-OD non-graph baseline run.
    cold_baseline_run_name: str = "cold_od_split_baselines_v1_notebook"
    cold_baseline_predictions_name: str = "predictions_cold_od_val_test.parquet"
    cold_baseline_all_splits_name: str = "predictions_cold_od_all_splits.parquet"

    # Split settings used only if the previous split file cannot be found.
    split_seed: int = 42
    val_pair_fraction: float = 0.10
    test_pair_fraction: float = 0.10
    train_years: Tuple[int, ...] = (2018, 2019, 2020, 2021, 2022)
    val_year: int = 2023
    test_year: int = 2024

    # Requested multi-seed protocol.
    seeds: Tuple[int, ...] = (7, 42, 123, 2026, 535)

    # Optimization settings.
    run_graphsage: bool = True
    run_hgt: bool = True
    max_epochs: int = 600
    patience: int = 80
    batch_size: int = 4096
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip_norm: float = 5.0
    lambda_iqr: float = 0.10
    eval_every: int = 1

    # Model settings.
    hidden_dim: int = 128
    graph_layers: int = 2
    hgt_heads: int = 4
    dropout: float = 0.10
    edge_hidden_dim: int = 128
    year_embedding_dim: int = 16

    # Training weights.
    use_sample_weight: bool = True
    weight_column: str = "obs_weight_sum"
    weight_clip_min: float = 0.05
    weight_clip_max: float = 20.0

    # Plotting and artifact options.
    make_plots: bool = True
    overwrite: bool = True
    save_model_checkpoints: bool = True

    # If True, average predictions over the five seeds for each model/checkpoint.
    build_seed_ensembles: bool = True


cfg = ExperimentConfig()

project_root = cfg.data_root
supervised_path = project_root / cfg.supervised_relative_path
manifest_path = project_root / cfg.manifest_relative_path
cold_baseline_dir = project_root / "10_experiments" / cfg.cold_baseline_run_name / cfg.scope
cold_baseline_predictions_path = cold_baseline_dir / cfg.cold_baseline_predictions_name
cold_baseline_all_splits_path = cold_baseline_dir / cfg.cold_baseline_all_splits_name

output_dir = project_root / "10_experiments" / cfg.run_name / cfg.scope
models_dir = output_dir / "models"
tables_dir = output_dir / "tables"
plots_dir = output_dir / "plots"
reports_dir = output_dir / "reports"

for path in [output_dir, models_dir, tables_dir, plots_dir, reports_dir]:
    path.mkdir(parents=True, exist_ok=True)

print("Supervised table:", supervised_path)
print("Feature manifest:", manifest_path)
print("Previous Cold-OD predictions:", cold_baseline_predictions_path)
print("Output directory:", output_dir)
print("Seeds:", cfg.seeds)

Supervised table: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet
Feature manifest: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\_metadata\freight_mnet_supervised_feature_manifest.json
Previous Cold-OD predictions: E:\NetworkOptimization\pythonProject1\Data\10_experiments\cold_od_split_baselines_v1_notebook\east_plus_gulf\predictions_cold_od_val_test.parquet
Output directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\graphsage_hgt_cold_od_baselines_v2_notebook\east_plus_gulf
Seeds: (7, 42, 123, 2026, 535)


## 3. Constants and reproducibility utilities

This cell defines shared constants and lightweight utilities.  FAF zone identifiers are normalized as strings because parquet/CSV readers may infer the same code as an integer in one file and a string in another.

In [3]:
LABEL_COLUMNS = ["truck_q25", "truck_q50", "truck_q75"]
QUANTILE_NAMES = ["q25", "q50", "q75"]
TAU_VALUES = np.array([0.25, 0.50, 0.75], dtype=np.float32)
TAUS_TENSOR = torch.tensor(TAU_VALUES, dtype=torch.float32)

DEFAULT_ORIGIN_CANDIDATES = ["faf_orig", "orig_faf", "origin_faf", "origin", "orig"]
DEFAULT_DEST_CANDIDATES = ["faf_dest", "dest_faf", "destination_faf", "destination", "dest"]
DEFAULT_YEAR_CANDIDATES = ["year", "Year"]

CHECKPOINT_METRICS = ["best_val_pinball", "best_val_q75", "best_val_iqr"]


def set_global_seed(seed: int) -> None:
    """Set Python, NumPy, and PyTorch seeds."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Determinism is helpful for reproducibility, but not all CUDA kernels are deterministic.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False


def format_faf_zone(value: Any) -> str:
    """Convert a FAF zone identifier to a stable three-digit string when possible."""
    if pd.isna(value):
        return "NA"
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    if text.isdigit():
        return f"{int(text):03d}"
    return text


def find_first_existing_column(df: pd.DataFrame, candidates: Sequence[str], role: str) -> str:
    """Return the first candidate column found in a dataframe."""
    for column in candidates:
        if column in df.columns:
            return column
    raise KeyError(f"Could not find {role} column. Candidates: {candidates}")


def ensure_columns(df: pd.DataFrame, columns: Sequence[str], context: str) -> None:
    """Raise a clear error if required columns are missing."""
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in {context}: {missing}")


def stable_hash_to_int(text: str, seed: int = 0) -> int:
    """Convert a string into a deterministic integer hash."""
    payload = f"{seed}::{text}".encode("utf-8")
    return int(hashlib.md5(payload).hexdigest()[:12], 16)


def to_float32_array(frame: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
    """Convert selected columns to a float32 NumPy array."""
    return frame[list(columns)].astype(float).to_numpy(dtype=np.float32)

## 4. Load supervised data and feature manifest

The supervised table is the model-ready FAF OD-year dataset.  The feature manifest is the source of truth for predictive tabular features.  This notebook does **not** add FMI aggregation diagnostics such as `obs_weight_sum` or `n_fmi_county_pairs` into the predictive feature vector.  Those columns are used only for loss weighting and diagnostics.

In [4]:
def load_supervised_table(path: Path) -> Tuple[pd.DataFrame, str, str, str]:
    """Load the supervised table and normalize key columns."""
    if not path.exists():
        raise FileNotFoundError(f"Supervised parquet not found: {path}")

    df_local = pd.read_parquet(path)
    origin_col_local = find_first_existing_column(df_local, DEFAULT_ORIGIN_CANDIDATES, "origin")
    dest_col_local = find_first_existing_column(df_local, DEFAULT_DEST_CANDIDATES, "destination")
    year_col_local = find_first_existing_column(df_local, DEFAULT_YEAR_CANDIDATES, "year")

    ensure_columns(df_local, LABEL_COLUMNS, "supervised table")

    df_local = df_local.copy()
    df_local[origin_col_local] = df_local[origin_col_local].map(format_faf_zone)
    df_local[dest_col_local] = df_local[dest_col_local].map(format_faf_zone)
    df_local[year_col_local] = df_local[year_col_local].astype(int)
    df_local["od_pair"] = df_local[origin_col_local].astype(str) + "->" + df_local[dest_col_local].astype(str)

    # Force label columns to numeric float values.
    for column in LABEL_COLUMNS:
        df_local[column] = pd.to_numeric(df_local[column], errors="coerce").astype(float)
    if df_local[LABEL_COLUMNS].isna().any().any():
        raise ValueError("Label columns contain NaN after numeric conversion.")

    return df_local, origin_col_local, dest_col_local, year_col_local


def load_feature_manifest(path: Path) -> List[str]:
    """Load predictive feature columns from the feature manifest."""
    if not path.exists():
        raise FileNotFoundError(f"Feature manifest not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        manifest = json.load(f)

    candidates = [
        manifest.get("feature_columns"),
        manifest.get("manifest_feature_columns"),
        manifest.get("features"),
    ]
    for candidate in candidates:
        if isinstance(candidate, list) and candidate:
            return [str(col) for col in candidate]
    raise KeyError("Could not find a non-empty feature column list in the manifest.")


def leakage_guard(feature_columns: Sequence[str]) -> None:
    """Ensure predictive features do not include labels or FMI diagnostics."""
    forbidden_exact = set(LABEL_COLUMNS) | {
        "obs_weight_sum",
        "n_fmi_county_pairs",
        "input_q25_min",
        "input_q25_max",
        "input_q50_min",
        "input_q50_max",
        "input_q75_min",
        "input_q75_max",
    }
    forbidden_substrings = ["truck_q25", "truck_q50", "truck_q75", "fmi", "movement", "crossing"]

    violations = []
    for column in feature_columns:
        lower = column.lower()
        if column in forbidden_exact or any(token in lower for token in forbidden_substrings):
            violations.append(column)
    if violations:
        raise ValueError(f"Potential leakage features found: {violations}")


df, origin_col, dest_col, year_col = load_supervised_table(supervised_path)
manifest_feature_columns = load_feature_manifest(manifest_path)
manifest_feature_columns = [column for column in manifest_feature_columns if column in df.columns]
leakage_guard(manifest_feature_columns)

print("Data shape:", df.shape)
print("Origin column:", origin_col)
print("Destination column:", dest_col)
print("Year column:", year_col)
print("Manifest feature count:", len(manifest_feature_columns))
print(df[year_col].value_counts().sort_index())

Data shape: (73972, 93)
Origin column: faf_orig
Destination column: faf_dest
Year column: year
Manifest feature count: 64
year
2018     9936
2019    10704
2020    10761
2021    10721
2022    10651
2023    10625
2024    10574
Name: count, dtype: int64


## 5. Basic label and schema checks

The labels should already be monotone because the preprocessing pipeline created `truck_q25 <= truck_q50 <= truck_q75`.  The graph models also use a monotone quantile head, so both labels and predictions should preserve quantile order.

In [5]:
def check_label_monotonicity(frame: pd.DataFrame) -> None:
    """Validate q25 <= q50 <= q75 for all supervised labels."""
    bad_q25_q50 = int((frame["truck_q25"] > frame["truck_q50"]).sum())
    bad_q50_q75 = int((frame["truck_q50"] > frame["truck_q75"]).sum())
    print("q25 > q50 violations:", bad_q25_q50)
    print("q50 > q75 violations:", bad_q50_q75)
    if bad_q25_q50 or bad_q50_q75:
        raise ValueError("Label monotonicity violations were found.")


check_label_monotonicity(df)

q25 > q50 violations: 0
q50 > q75 violations: 0


## 6. Load or reconstruct the Cold-OD split

The preferred path is to reuse the previous Cold-OD split from `predictions_cold_od_all_splits.parquet`.  That file contains the row keys and split labels used by the non-graph Cold-OD baselines.  Reusing it guarantees that GraphSAGE/HGT are evaluated against MLP/LightGBM on the same unseen OD pairs.

If the previous file is not available, this cell constructs a deterministic OD-pair-level split.  In both cases, the notebook enforces that train, validation, and test OD-pair sets are disjoint.

In [6]:
def normalize_prediction_key_columns(
    pred_df: pd.DataFrame,
    origin_col_target: str,
    dest_col_target: str,
    year_col_target: str,
) -> pd.DataFrame:
    """Normalize key columns in an external prediction dataframe."""
    result = pred_df.copy()
    p_origin = find_first_existing_column(result, DEFAULT_ORIGIN_CANDIDATES, "prediction origin")
    p_dest = find_first_existing_column(result, DEFAULT_DEST_CANDIDATES, "prediction destination")
    p_year = find_first_existing_column(result, DEFAULT_YEAR_CANDIDATES, "prediction year")

    result[origin_col_target] = result[p_origin].map(format_faf_zone)
    result[dest_col_target] = result[p_dest].map(format_faf_zone)
    result[year_col_target] = result[p_year].astype(int)
    result["od_pair"] = result[origin_col_target].astype(str) + "->" + result[dest_col_target].astype(str)
    return result


def load_cold_split_from_predictions(
    supervised_df: pd.DataFrame,
    predictions_path: Path,
    origin_col_name: str,
    dest_col_name: str,
    year_col_name: str,
) -> Optional[pd.Series]:
    """Infer row-level Cold-OD split labels from a previous prediction parquet."""
    if not predictions_path.exists():
        return None

    pred_df = pd.read_parquet(predictions_path)
    if "split" not in pred_df.columns:
        return None

    pred_df = normalize_prediction_key_columns(pred_df, origin_col_name, dest_col_name, year_col_name)
    split_keys = pred_df[[origin_col_name, dest_col_name, year_col_name, "split"]].drop_duplicates()

    duplicate_key_counts = split_keys.groupby([origin_col_name, dest_col_name, year_col_name])["split"].nunique()
    if int((duplicate_key_counts > 1).sum()) > 0:
        raise ValueError("Previous Cold-OD prediction file has conflicting split labels for the same row key.")

    merged = supervised_df[[origin_col_name, dest_col_name, year_col_name]].merge(
        split_keys,
        on=[origin_col_name, dest_col_name, year_col_name],
        how="left",
    )
    split = merged["split"].fillna("unused").astype(str)
    return split


def build_deterministic_cold_od_split(
    supervised_df: pd.DataFrame,
    origin_col_name: str,
    dest_col_name: str,
    year_col_name: str,
    config: ExperimentConfig,
) -> pd.Series:
    """Build a deterministic Cold-OD split if previous split artifacts are unavailable."""
    rng = np.random.default_rng(config.split_seed)
    all_pairs = np.array(sorted(supervised_df["od_pair"].unique()))

    train_year_mask = supervised_df[year_col_name].isin(config.train_years)
    val_year_mask = supervised_df[year_col_name].eq(config.val_year)
    test_year_mask = supervised_df[year_col_name].eq(config.test_year)

    train_candidates = set(supervised_df.loc[train_year_mask, "od_pair"].unique())
    val_candidates = sorted(set(supervised_df.loc[val_year_mask, "od_pair"].unique()))
    test_candidates = sorted(set(supervised_df.loc[test_year_mask, "od_pair"].unique()))

    n_test = max(1, int(round(len(test_candidates) * config.test_pair_fraction)))
    test_pairs = set(rng.choice(test_candidates, size=n_test, replace=False).tolist())

    remaining_val_candidates = [pair for pair in val_candidates if pair not in test_pairs]
    n_val = max(1, int(round(len(remaining_val_candidates) * config.val_pair_fraction)))
    val_pairs = set(rng.choice(remaining_val_candidates, size=n_val, replace=False).tolist())

    train_pairs = train_candidates - val_pairs - test_pairs

    split = pd.Series("unused", index=supervised_df.index, dtype="object")
    split.loc[train_year_mask & supervised_df["od_pair"].isin(train_pairs)] = "train"
    split.loc[val_year_mask & supervised_df["od_pair"].isin(val_pairs)] = "val"
    split.loc[test_year_mask & supervised_df["od_pair"].isin(test_pairs)] = "test"
    return split.astype(str)


def assert_cold_od_integrity(frame: pd.DataFrame, split_col: str) -> None:
    """Assert that train/val/test OD pairs are disjoint."""
    train_pairs = set(frame.loc[frame[split_col].eq("train"), "od_pair"])
    val_pairs = set(frame.loc[frame[split_col].eq("val"), "od_pair"])
    test_pairs = set(frame.loc[frame[split_col].eq("test"), "od_pair"])

    overlaps = {
        "train_val": len(train_pairs & val_pairs),
        "train_test": len(train_pairs & test_pairs),
        "val_test": len(val_pairs & test_pairs),
    }
    print("Cold-OD pair overlaps:", overlaps)
    if any(count > 0 for count in overlaps.values()):
        raise ValueError(f"Cold-OD split leakage detected: {overlaps}")


split_series = load_cold_split_from_predictions(
    df,
    cold_baseline_all_splits_path,
    origin_col,
    dest_col,
    year_col,
)

if split_series is None:
    print("Previous Cold-OD split not found. Building a deterministic split.")
    split_series = build_deterministic_cold_od_split(df, origin_col, dest_col, year_col, cfg)
else:
    print("Loaded Cold-OD split from previous baseline predictions.")

df["cold_split"] = split_series.values
assert_cold_od_integrity(df, "cold_split")

split_summary = (
    df.groupby("cold_split")
    .agg(
        n_rows=("od_pair", "size"),
        n_od_pairs=("od_pair", "nunique"),
        min_year=(year_col, "min"),
        max_year=(year_col, "max"),
    )
    .reset_index()
    .sort_values("cold_split")
)
print(split_summary)

train_mask_np = df["cold_split"].eq("train").to_numpy()
val_mask_np = df["cold_split"].eq("val").to_numpy()
test_mask_np = df["cold_split"].eq("test").to_numpy()

if not train_mask_np.any() or not val_mask_np.any() or not test_mask_np.any():
    raise ValueError("Cold-OD split must contain non-empty train, validation, and test rows.")

Loaded Cold-OD split from previous baseline predictions.
Cold-OD pair overlaps: {'train_val': 0, 'train_test': 0, 'val_test': 0}
  cold_split  n_rows  n_od_pairs  min_year  max_year
0       test    1057        1057      2024      2024
1      train   42849        8748      2018      2022
2     unused   29109       10631      2018      2024
3        val     957         957      2023      2023


## 7. Sample weights

`obs_weight_sum` is used only as a loss/evaluation weight.  It is not used as a predictive feature because it is an FMI aggregation diagnostic rather than an exogenous freight feature.

In [7]:
def build_sample_weights(frame: pd.DataFrame, split_col: str, config: ExperimentConfig) -> np.ndarray:
    """Build clipped and train-normalized sample weights."""
    if not config.use_sample_weight or config.weight_column not in frame.columns:
        return np.ones(len(frame), dtype=np.float32)

    raw = pd.to_numeric(frame[config.weight_column], errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)
    weights = np.log1p(np.maximum(raw, 0.0))
    weights = np.clip(weights, config.weight_clip_min, config.weight_clip_max)

    train_mask = frame[split_col].eq("train").to_numpy()
    train_mean = float(weights[train_mask].mean()) if train_mask.any() else float(weights.mean())
    if not np.isfinite(train_mean) or train_mean <= 0:
        train_mean = 1.0
    return (weights / train_mean).astype(np.float32)


df["sample_weight"] = build_sample_weights(df, "cold_split", cfg)
print("Sample weight summary:")
print(df.loc[df["cold_split"].isin(["train", "val", "test"]), "sample_weight"].describe())

Sample weight summary:
count    44863.000000
mean         0.998810
std          0.339039
min          0.124688
25%          0.758393
50%          1.003247
75%          1.243303
max          2.062156
Name: sample_weight, dtype: float64


## 8. Cold-OD fallback priors

Exact OD historical priors are not allowed in Cold-OD because held-out OD pairs are unseen during training.  Instead, the graph models use a fallback prior based on training-only origin and destination medians.

For training rows, this notebook uses an out-of-fold origin/destination fallback prior.  This prevents a training row from directly contributing to its own base prior.

In [8]:
COLD_PRIOR_COLUMNS = [
    "cold_prior_truck_q25",
    "cold_prior_truck_q50",
    "cold_prior_truck_q75",
    "cold_prior_iqr",
    "cold_prior_q75_q50_gap",
    "cold_prior_q50_q25_gap",
    "cold_has_origin_prior",
    "cold_has_dest_prior",
    "cold_has_any_zone_prior",
]


def fit_zone_prior_maps(
    fit_df: pd.DataFrame,
    origin_col_name: str,
    dest_col_name: str,
) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray], np.ndarray]:
    """Fit origin and destination median maps from a training subset."""
    if fit_df.empty:
        raise ValueError("Cannot fit zone priors from an empty dataframe.")

    global_prior = fit_df[LABEL_COLUMNS].median().to_numpy(dtype=np.float32)
    origin_map = {
        str(key): values.to_numpy(dtype=np.float32)
        for key, values in fit_df.groupby(origin_col_name)[LABEL_COLUMNS].median().iterrows()
    }
    dest_map = {
        str(key): values.to_numpy(dtype=np.float32)
        for key, values in fit_df.groupby(dest_col_name)[LABEL_COLUMNS].median().iterrows()
    }
    return origin_map, dest_map, global_prior


def apply_zone_priors(
    target_df: pd.DataFrame,
    origin_map: Mapping[str, np.ndarray],
    dest_map: Mapping[str, np.ndarray],
    global_prior: np.ndarray,
    origin_col_name: str,
    dest_col_name: str,
) -> pd.DataFrame:
    """Apply fitted origin/destination prior maps to target rows."""
    priors = np.zeros((len(target_df), 3), dtype=np.float32)
    has_origin = np.zeros(len(target_df), dtype=np.float32)
    has_dest = np.zeros(len(target_df), dtype=np.float32)

    for i, (orig, dest) in enumerate(zip(target_df[origin_col_name].astype(str), target_df[dest_col_name].astype(str))):
        parts: List[np.ndarray] = []
        if orig in origin_map:
            parts.append(origin_map[orig])
            has_origin[i] = 1.0
        if dest in dest_map:
            parts.append(dest_map[dest])
            has_dest[i] = 1.0
        if parts:
            priors[i] = np.mean(np.stack(parts, axis=0), axis=0)
        else:
            priors[i] = global_prior

    # The average of monotone quantile vectors is monotone, but sorting is a safe guard.
    priors = np.sort(priors, axis=1)

    result = pd.DataFrame(index=target_df.index)
    result["cold_prior_truck_q25"] = priors[:, 0]
    result["cold_prior_truck_q50"] = priors[:, 1]
    result["cold_prior_truck_q75"] = priors[:, 2]
    result["cold_prior_iqr"] = priors[:, 2] - priors[:, 0]
    result["cold_prior_q75_q50_gap"] = priors[:, 2] - priors[:, 1]
    result["cold_prior_q50_q25_gap"] = priors[:, 1] - priors[:, 0]
    result["cold_has_origin_prior"] = has_origin
    result["cold_has_dest_prior"] = has_dest
    result["cold_has_any_zone_prior"] = np.maximum(has_origin, has_dest)
    return result


def build_oof_cold_priors(
    frame: pd.DataFrame,
    origin_col_name: str,
    dest_col_name: str,
    split_col: str,
    n_folds: int = 5,
) -> pd.DataFrame:
    """Build out-of-fold fallback priors for train rows and train-only priors for other rows."""
    output = pd.DataFrame(index=frame.index, columns=COLD_PRIOR_COLUMNS, dtype=np.float32)
    train_indices = frame.index[frame[split_col].eq("train")].to_numpy()
    train_df = frame.loc[train_indices].copy()

    # Fold assignment is by OD pair, so all training years of the same pair share the same fold.
    fold_for_pair = {
        pair: stable_hash_to_int(pair, seed=cfg.split_seed) % n_folds
        for pair in train_df["od_pair"].unique()
    }
    train_folds = train_df["od_pair"].map(fold_for_pair).astype(int)

    for fold_id in range(n_folds):
        fit_part = train_df.loc[train_folds.ne(fold_id)]
        target_part = train_df.loc[train_folds.eq(fold_id)]
        if target_part.empty:
            continue
        origin_map, dest_map, global_prior = fit_zone_prior_maps(fit_part, origin_col_name, dest_col_name)
        output.loc[target_part.index, COLD_PRIOR_COLUMNS] = apply_zone_priors(
            target_part, origin_map, dest_map, global_prior, origin_col_name, dest_col_name
        )[COLD_PRIOR_COLUMNS].to_numpy(dtype=np.float32)

    # Validation, test, and unused rows use priors fitted on all training rows only.
    origin_map, dest_map, global_prior = fit_zone_prior_maps(train_df, origin_col_name, dest_col_name)
    non_train = frame.index[~frame[split_col].eq("train")].to_numpy()
    if len(non_train) > 0:
        output.loc[non_train, COLD_PRIOR_COLUMNS] = apply_zone_priors(
            frame.loc[non_train], origin_map, dest_map, global_prior, origin_col_name, dest_col_name
        )[COLD_PRIOR_COLUMNS].to_numpy(dtype=np.float32)

    if output.isna().any().any():
        raise ValueError("Cold fallback prior construction produced NaN values.")
    return output.astype(np.float32)


prior_features = build_oof_cold_priors(df, origin_col, dest_col, "cold_split", n_folds=5)
for column in COLD_PRIOR_COLUMNS:
    df[column] = prior_features[column].astype(np.float32)

print(df.loc[df["cold_split"].isin(["train", "val", "test"]), COLD_PRIOR_COLUMNS].describe().T)

                           count         mean         std          min  \
cold_prior_truck_q25     44863.0  1558.371094  200.544373  1116.939941   
cold_prior_truck_q50     44863.0  2315.859131  306.046021  1602.132568   
cold_prior_truck_q75     44863.0  3714.135498  461.039520  2724.810059   
cold_prior_iqr           44863.0  2155.764648  315.572876  1493.565063   
cold_prior_q75_q50_gap   44863.0  1398.276733  209.684814   762.229980   
cold_prior_q50_q25_gap   44863.0   757.487793  138.019714   378.612427   
cold_has_origin_prior    44863.0     1.000000    0.000000     1.000000   
cold_has_dest_prior      44863.0     1.000000    0.000000     1.000000   
cold_has_any_zone_prior  44863.0     1.000000    0.000000     1.000000   

                                 25%          50%          75%          max  
cold_prior_truck_q25     1403.614990  1534.112549  1689.675049  2387.107422  
cold_prior_truck_q50     2099.617432  2308.905029  2532.462402  3379.750000  
cold_prior_truck_q75     

## 9. Node features from training rows only

FAF node features are computed from training rows only.  This avoids using validation/test labels or validation/test OD-specific information to build node embeddings.  The features include origin-role aggregates, destination-role aggregates, and training graph degree statistics.

In [9]:
NODE_AGG_FEATURES = [
    "total_tons_modes_1_2_5",
    "total_tmiles_modes_1_2_5",
    "total_value_modes_1_2_5",
    "tons_truck",
    "tons_rail",
    "tons_multimodal",
    "tmiles_truck",
    "tmiles_rail",
    "tmiles_multimodal",
    "value_truck",
    "value_rail",
    "value_multimodal",
    "n_modes_with_positive_tons",
    "has_truck_demand",
    "has_rail_demand",
    "has_multimodal_demand",
]


def safe_existing(columns: Sequence[str], frame: pd.DataFrame) -> List[str]:
    """Return the subset of columns present in a dataframe."""
    return [column for column in columns if column in frame.columns]


def aggregate_role_features(
    train_df: pd.DataFrame,
    zone_col: str,
    numeric_features: Sequence[str],
    prefix: str,
) -> pd.DataFrame:
    """Aggregate numeric row features for zones in a given role."""
    zones = train_df[[zone_col]].drop_duplicates().rename(columns={zone_col: "faf_zone"})
    if not numeric_features:
        return zones
    agg = train_df.groupby(zone_col)[list(numeric_features)].mean().reset_index()
    agg = agg.rename(columns={zone_col: "faf_zone"})
    agg = agg.rename(columns={column: f"{prefix}_{column}" for column in numeric_features})
    return agg


def build_node_feature_frame(
    frame: pd.DataFrame,
    origin_col_name: str,
    dest_col_name: str,
    split_col: str,
) -> pd.DataFrame:
    """Build FAF-zone node features using training rows only."""
    train_df = frame.loc[frame[split_col].eq("train")].copy()
    all_zones = sorted(set(frame[origin_col_name].astype(str)) | set(frame[dest_col_name].astype(str)))
    node_df = pd.DataFrame({"faf_zone": all_zones})

    numeric_features = safe_existing(NODE_AGG_FEATURES, frame)
    origin_agg = aggregate_role_features(train_df, origin_col_name, numeric_features, "as_origin")
    dest_agg = aggregate_role_features(train_df, dest_col_name, numeric_features, "as_dest")

    node_df = node_df.merge(origin_agg, on="faf_zone", how="left")
    node_df = node_df.merge(dest_agg, on="faf_zone", how="left")

    out_degree = train_df.groupby(origin_col_name).size().rename("train_out_degree").reset_index()
    out_degree = out_degree.rename(columns={origin_col_name: "faf_zone"})
    in_degree = train_df.groupby(dest_col_name).size().rename("train_in_degree").reset_index()
    in_degree = in_degree.rename(columns={dest_col_name: "faf_zone"})

    node_df = node_df.merge(out_degree, on="faf_zone", how="left")
    node_df = node_df.merge(in_degree, on="faf_zone", how="left")
    node_df["train_out_degree"] = node_df["train_out_degree"].fillna(0.0)
    node_df["train_in_degree"] = node_df["train_in_degree"].fillna(0.0)
    node_df["train_total_degree"] = node_df["train_out_degree"] + node_df["train_in_degree"]

    node_df["faf_zone_numeric"] = pd.to_numeric(node_df["faf_zone"], errors="coerce").fillna(0.0)

    numeric_cols = [col for col in node_df.columns if col != "faf_zone"]
    fill_values = node_df[numeric_cols].median(numeric_only=True).fillna(0.0)
    node_df[numeric_cols] = node_df[numeric_cols].fillna(fill_values).astype(np.float32)

    return node_df


def fit_node_scaler(node_df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Fit a simple z-score scaler for node features."""
    feature_cols = [column for column in node_df.columns if column != "faf_zone"]
    arr = node_df[feature_cols].to_numpy(dtype=np.float32)
    mean = arr.mean(axis=0)
    std = arr.std(axis=0)
    std[std < 1e-6] = 1.0
    return mean.astype(np.float32), std.astype(np.float32), feature_cols


node_feature_df = build_node_feature_frame(df, origin_col, dest_col, "cold_split")
node_mean, node_std, node_feature_columns = fit_node_scaler(node_feature_df)
node_features_np = ((node_feature_df[node_feature_columns].to_numpy(dtype=np.float32) - node_mean) / node_std).astype(np.float32)
node_features_np[~np.isfinite(node_features_np)] = 0.0

zone_to_idx = {zone: idx for idx, zone in enumerate(node_feature_df["faf_zone"].astype(str).tolist())}
print("Number of FAF nodes:", len(zone_to_idx))
print("Node feature dimension:", node_features_np.shape[1])

Number of FAF nodes: 104
Node feature dimension: 36


## 10. Numeric edge-feature preprocessing

The graph decoder receives OD edge features.  These include the 64 manifest features plus Cold-OD fallback-prior features.  The imputer and scaler are fitted on training rows only.

In [10]:
@dataclass
class NumericPreprocessor:
    """Median imputer plus z-score scaler for numeric edge features."""

    columns: List[str]
    medians: Dict[str, float]
    means: Dict[str, float]
    stds: Dict[str, float]

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        """Transform a dataframe into a float32 design matrix."""
        result = np.zeros((len(frame), len(self.columns)), dtype=np.float32)
        for j, column in enumerate(self.columns):
            if column in frame.columns:
                values = pd.to_numeric(frame[column], errors="coerce")
            else:
                values = pd.Series(np.nan, index=frame.index)
            arr = values.fillna(self.medians[column]).to_numpy(dtype=np.float32)
            arr = (arr - self.means[column]) / self.stds[column]
            arr[~np.isfinite(arr)] = 0.0
            result[:, j] = arr.astype(np.float32)
        return result

    def to_dict(self) -> Dict[str, Any]:
        """Serialize the preprocessor to a JSON-compatible dictionary."""
        return {
            "columns": self.columns,
            "medians": self.medians,
            "means": self.means,
            "stds": self.stds,
        }


def fit_numeric_preprocessor(frame: pd.DataFrame, columns: Sequence[str]) -> NumericPreprocessor:
    """Fit a median imputer and z-score scaler on training rows."""
    medians: Dict[str, float] = {}
    means: Dict[str, float] = {}
    stds: Dict[str, float] = {}
    for column in columns:
        if column in frame.columns:
            values = pd.to_numeric(frame[column], errors="coerce")
        else:
            values = pd.Series(np.nan, index=frame.index)
        median = float(values.median()) if values.notna().any() else 0.0
        filled = values.fillna(median).to_numpy(dtype=np.float32)
        mean = float(np.mean(filled))
        std = float(np.std(filled))
        if not np.isfinite(std) or std < 1e-6:
            std = 1.0
        medians[column] = median
        means[column] = mean
        stds[column] = std
    return NumericPreprocessor(columns=list(columns), medians=medians, means=means, stds=stds)


edge_feature_columns = list(manifest_feature_columns) + COLD_PRIOR_COLUMNS
edge_feature_columns = [column for column in edge_feature_columns if column in df.columns]
train_df_for_preprocessor = df.loc[df["cold_split"].eq("train")].copy()
edge_preprocessor = fit_numeric_preprocessor(train_df_for_preprocessor, edge_feature_columns)
edge_features_np = edge_preprocessor.transform(df)

print("Edge feature dimension:", edge_features_np.shape[1])
print("First 10 edge features:", edge_feature_columns[:10])

Edge feature dimension: 73
First 10 edge features: ['dest_east_plus_gulf_county_share', 'dest_east_plus_gulf_faf_flag_any_county', 'dest_max_county_centroid_lon', 'dest_min_county_centroid_lon', 'dest_n_counties', 'dest_n_east_plus_gulf_counties', 'has_multimodal_demand', 'has_rail_demand', 'has_truck_demand', 'log1p_tmiles_multimodal']


## 11. Build graph edges and tensors

The message-passing graph is constructed from training rows only.  Validation and test supervised OD pairs are not included as message-passing edges, which prevents Cold-OD leakage.

GraphSAGE receives a homogeneous edge index containing all training graph edges.  HGT receives a typed edge-index dictionary with separate relation names.

In [11]:
def make_edge_index_from_pairs(pairs: Iterable[Tuple[str, str]], zone_index: Mapping[str, int]) -> torch.Tensor:
    """Build a PyTorch edge_index tensor from zone-code pairs."""
    src: List[int] = []
    dst: List[int] = []
    for orig, dest in pairs:
        if orig in zone_index and dest in zone_index:
            src.append(zone_index[orig])
            dst.append(zone_index[dest])
    if not src:
        return torch.empty((2, 0), dtype=torch.long)
    return torch.tensor([src, dst], dtype=torch.long)


def unique_pairs_from_mask(frame: pd.DataFrame, mask: np.ndarray, origin_col_name: str, dest_col_name: str) -> List[Tuple[str, str]]:
    """Return sorted unique origin-destination pairs for rows selected by a mask."""
    part = frame.loc[mask, [origin_col_name, dest_col_name]].drop_duplicates()
    return list(map(tuple, part.astype(str).to_numpy()))


def build_typed_edges(frame: pd.DataFrame, origin_col_name: str, dest_col_name: str, split_col: str, zone_index: Mapping[str, int]) -> Dict[Tuple[str, str, str], torch.Tensor]:
    """Build typed HGT edges from training rows only."""
    train_part = frame.loc[frame[split_col].eq("train")].copy()
    edge_dict: Dict[Tuple[str, str, str], torch.Tensor] = {}

    def add_relation(name: str, relation_mask: np.ndarray) -> None:
        pairs = unique_pairs_from_mask(train_part, relation_mask, origin_col_name, dest_col_name)
        forward = make_edge_index_from_pairs(pairs, zone_index)
        reverse = torch.stack([forward[1], forward[0]], dim=0) if forward.numel() > 0 else forward.clone()
        edge_dict[("faf", name, "faf")] = forward
        edge_dict[("faf", f"rev_{name}", "faf")] = reverse

    # All training OD pairs define the baseline observed training relation.
    add_relation("train_od", np.ones(len(train_part), dtype=bool))

    # Demand-mode relations use mode flags when available. If a flag is unavailable, the relation is skipped.
    relation_columns = {
        "demand_truck": "has_truck_demand",
        "demand_rail": "has_rail_demand",
        "demand_multimodal": "has_multimodal_demand",
    }
    for relation_name, flag_col in relation_columns.items():
        if flag_col in train_part.columns:
            flag_values = pd.to_numeric(train_part[flag_col], errors="coerce").fillna(0.0).to_numpy()
            add_relation(relation_name, flag_values > 0)

    # Self-loops ensure every node can retain its own state through HGTConv.
    n_nodes = len(zone_index)
    self_edges = torch.arange(n_nodes, dtype=torch.long).unsqueeze(0).repeat(2, 1)
    edge_dict[("faf", "self_loop", "faf")] = self_edges
    return edge_dict


def build_homogeneous_edge_index(edge_dict: Mapping[Tuple[str, str, str], torch.Tensor]) -> torch.Tensor:
    """Merge all typed edges into one homogeneous GraphSAGE edge index."""
    edge_indices = [edge for edge in edge_dict.values() if edge.numel() > 0]
    if not edge_indices:
        return torch.empty((2, 0), dtype=torch.long)
    merged = torch.cat(edge_indices, dim=1)
    return torch.unique(merged, dim=1)


node_x_cpu = torch.tensor(node_features_np, dtype=torch.float32)
typed_edge_index_dict_cpu = build_typed_edges(df, origin_col, dest_col, "cold_split", zone_to_idx)
hom_edge_index_cpu = build_homogeneous_edge_index(typed_edge_index_dict_cpu)

print("Typed edge counts:")
for edge_type, edge_index in typed_edge_index_dict_cpu.items():
    print(f"  {edge_type}: {edge_index.shape[1]}")
print("Homogeneous edge count:", hom_edge_index_cpu.shape[1])

Typed edge counts:
  ('faf', 'train_od', 'faf'): 8748
  ('faf', 'rev_train_od', 'faf'): 8748
  ('faf', 'demand_truck', 'faf'): 8693
  ('faf', 'rev_demand_truck', 'faf'): 8693
  ('faf', 'demand_rail', 'faf'): 5834
  ('faf', 'rev_demand_rail', 'faf'): 5834
  ('faf', 'demand_multimodal', 'faf'): 8692
  ('faf', 'rev_demand_multimodal', 'faf'): 8692
  ('faf', 'self_loop', 'faf'): 104
Homogeneous edge count: 10412


## 12. Build supervised edge tensors

Every OD-year row becomes a supervised edge-label example.  The graph structure is static within this baseline, but the decoder receives OD-year edge attributes, fallback priors, and a year embedding.

In [12]:
def build_year_index(frame: pd.DataFrame, year_col_name: str) -> Tuple[np.ndarray, Dict[int, int]]:
    """Map years to compact integer indices for the year embedding."""
    years = sorted(frame[year_col_name].astype(int).unique().tolist())
    mapping = {year: i for i, year in enumerate(years)}
    return frame[year_col_name].astype(int).map(mapping).to_numpy(dtype=np.int64), mapping


def build_edge_label_index(frame: pd.DataFrame, origin_col_name: str, dest_col_name: str, zone_index: Mapping[str, int]) -> torch.Tensor:
    """Build edge_label_index for supervised OD-year rows."""
    src = frame[origin_col_name].astype(str).map(zone_index).to_numpy(dtype=np.int64)
    dst = frame[dest_col_name].astype(str).map(zone_index).to_numpy(dtype=np.int64)
    if np.isnan(src).any() or np.isnan(dst).any():
        raise ValueError("Some supervised rows have FAF zones missing from the node index.")
    return torch.tensor([src, dst], dtype=torch.long)


year_idx_np, year_to_idx = build_year_index(df, year_col)
y_np = df[LABEL_COLUMNS].to_numpy(dtype=np.float32)
base_prior_np = df[["cold_prior_truck_q25", "cold_prior_truck_q50", "cold_prior_truck_q75"]].to_numpy(dtype=np.float32)

# Use a fixed target scale to improve neural optimization while keeping metrics in minutes.
target_scale = float(np.nanmedian(y_np[train_mask_np, 1]))
if not np.isfinite(target_scale) or target_scale <= 0:
    target_scale = 1000.0
print("Target scale:", target_scale)

edge_label_index_cpu = build_edge_label_index(df, origin_col, dest_col, zone_to_idx)
edge_attr_cpu = torch.tensor(edge_features_np, dtype=torch.float32)
y_scaled_cpu = torch.tensor(y_np / target_scale, dtype=torch.float32)
base_prior_scaled_cpu = torch.tensor(base_prior_np / target_scale, dtype=torch.float32)
weights_cpu = torch.tensor(df["sample_weight"].to_numpy(dtype=np.float32), dtype=torch.float32)
year_idx_cpu = torch.tensor(year_idx_np, dtype=torch.long)

split_indices_cpu = {
    "train": torch.tensor(np.flatnonzero(train_mask_np), dtype=torch.long),
    "val": torch.tensor(np.flatnonzero(val_mask_np), dtype=torch.long),
    "test": torch.tensor(np.flatnonzero(test_mask_np), dtype=torch.long),
}

print("Split index sizes:", {k: int(v.numel()) for k, v in split_indices_cpu.items()})
print("Year mapping:", year_to_idx)

Target scale: 2317.030029296875
Split index sizes: {'train': 42849, 'val': 957, 'test': 1057}
Year mapping: {2018: 0, 2019: 1, 2020: 2, 2021: 3, 2022: 4, 2023: 5, 2024: 6}


C:\Users\Nick_James\AppData\Local\Temp\ipykernel_45016\4236578387.py:14: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  return torch.tensor([src, dst], dtype=torch.long)


## 13. Device placement

This cell moves the static graph tensors and supervised row tensors onto the active device.  The FAF-zone graph is small, so full-graph message passing is used; supervised edge labels are mini-batched.

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

node_x = node_x_cpu.to(device)
hom_edge_index = hom_edge_index_cpu.to(device)
typed_edge_index_dict = {edge_type: edge_index.to(device) for edge_type, edge_index in typed_edge_index_dict_cpu.items()}
edge_label_index = edge_label_index_cpu.to(device)
edge_attr = edge_attr_cpu.to(device)
y_scaled = y_scaled_cpu.to(device)
base_prior_scaled = base_prior_scaled_cpu.to(device)
weights = weights_cpu.to(device)
year_idx = year_idx_cpu.to(device)
split_indices = {name: tensor.to(device) for name, tensor in split_indices_cpu.items()}
TAUS = TAUS_TENSOR.to(device)

Using device: cuda


C:\Users\Nick_James\AppData\Local\Temp\ipykernel_45016\1381029407.py:4: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  node_x = node_x_cpu.to(device)


## 14. Loss and metric utilities

Training uses weighted pinball loss plus an optional IQR auxiliary loss.  Evaluation metrics are computed in minutes.  The same metric function is used for train/validation/test summaries and strict test-only diagnostics.

In [14]:
def inv_softplus(x: torch.Tensor) -> torch.Tensor:
    """Numerically stable inverse softplus for positive tensors."""
    x = torch.clamp(x, min=1e-6)
    return torch.where(x > 20.0, x, torch.log(torch.expm1(x)))


def torch_pinball_loss(pred: torch.Tensor, target: torch.Tensor, taus: torch.Tensor) -> torch.Tensor:
    """Return per-row, per-quantile pinball losses."""
    diff = target - pred
    return torch.maximum(taus * diff, (taus - 1.0) * diff)


def training_loss(pred_scaled: torch.Tensor, target_scaled: torch.Tensor, row_weights: torch.Tensor, lambda_iqr: float) -> torch.Tensor:
    """Compute weighted pinball loss plus optional IQR SmoothL1 loss."""
    pinball = torch_pinball_loss(pred_scaled, target_scaled, TAUS).mean(dim=1)
    loss = (pinball * row_weights).mean()
    if lambda_iqr > 0:
        pred_iqr = pred_scaled[:, 2] - pred_scaled[:, 0]
        true_iqr = target_scaled[:, 2] - target_scaled[:, 0]
        loss = loss + lambda_iqr * F.smooth_l1_loss(pred_iqr, true_iqr)
    return loss


def numpy_pinball(y_true: np.ndarray, y_pred: np.ndarray, taus: Sequence[float]) -> np.ndarray:
    """Return per-row, per-quantile pinball losses as a NumPy array."""
    taus_arr = np.asarray(taus, dtype=np.float32).reshape(1, -1)
    diff = y_true - y_pred
    return np.maximum(taus_arr * diff, (taus_arr - 1.0) * diff)


def compute_metrics_from_arrays(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    row_weights: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    """Compute FREIGHT-MNet quantile metrics in minutes."""
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    pinball = numpy_pinball(y_true, y_pred, TAU_VALUES)

    metrics: Dict[str, float] = {
        "n_rows": float(len(y_true)),
        "mae_q25": float(np.mean(np.abs(y_pred[:, 0] - y_true[:, 0]))),
        "mae_q50": float(np.mean(np.abs(y_pred[:, 1] - y_true[:, 1]))),
        "mae_q75": float(np.mean(np.abs(y_pred[:, 2] - y_true[:, 2]))),
        "rmse_q50": float(np.sqrt(np.mean((y_pred[:, 1] - y_true[:, 1]) ** 2))),
        "pinball_q25": float(np.mean(pinball[:, 0])),
        "pinball_q50": float(np.mean(pinball[:, 1])),
        "pinball_q75": float(np.mean(pinball[:, 2])),
        "pinball_mean": float(np.mean(pinball.mean(axis=1))),
        "iqr_mae": float(np.mean(np.abs((y_pred[:, 2] - y_pred[:, 0]) - (y_true[:, 2] - y_true[:, 0])))),
        "bias_q25": float(np.mean(y_pred[:, 0] - y_true[:, 0])),
        "bias_q50": float(np.mean(y_pred[:, 1] - y_true[:, 1])),
        "bias_q75": float(np.mean(y_pred[:, 2] - y_true[:, 2])),
        "pred_iqr_mean": float(np.mean(y_pred[:, 2] - y_pred[:, 0])),
        "true_iqr_mean": float(np.mean(y_true[:, 2] - y_true[:, 0])),
        "raw_crossing_rate": float(np.mean((y_pred[:, 0] > y_pred[:, 1]) | (y_pred[:, 1] > y_pred[:, 2]))),
        "q50_inside_pred_q25_q75_rate": float(np.mean((y_true[:, 1] >= y_pred[:, 0]) & (y_true[:, 1] <= y_pred[:, 2]))),
    }

    if row_weights is not None:
        w = np.asarray(row_weights, dtype=np.float32)
        if len(w) == len(y_true) and np.isfinite(w).all() and float(w.sum()) > 0:
            row_pinball = pinball.mean(axis=1)
            metrics["weighted_pinball_mean"] = float(np.average(row_pinball, weights=w))
        else:
            metrics["weighted_pinball_mean"] = metrics["pinball_mean"]
    else:
        metrics["weighted_pinball_mean"] = metrics["pinball_mean"]

    if len(y_true) > 0:
        q75_threshold = float(np.quantile(y_true[:, 2], 0.90))
        stress_mask = y_true[:, 2] >= q75_threshold
        metrics["stress_top10_threshold_q75"] = q75_threshold
        metrics["stress_top10_n_rows_q75"] = float(stress_mask.sum())
        metrics["stress_top10_mae_q75"] = float(np.mean(np.abs(y_pred[stress_mask, 2] - y_true[stress_mask, 2]))) if stress_mask.any() else np.nan

        iqr_true = y_true[:, 2] - y_true[:, 0]
        iqr_threshold = float(np.quantile(iqr_true, 0.90))
        top_iqr_mask = iqr_true >= iqr_threshold
        metrics["top_iqr10_threshold"] = iqr_threshold
        metrics["top_iqr10_n_rows"] = float(top_iqr_mask.sum())
        metrics["top_iqr10_mae_q75"] = float(np.mean(np.abs(y_pred[top_iqr_mask, 2] - y_true[top_iqr_mask, 2]))) if top_iqr_mask.any() else np.nan
        metrics["top_iqr10_iqr_mae"] = float(np.mean(np.abs((y_pred[top_iqr_mask, 2] - y_pred[top_iqr_mask, 0]) - iqr_true[top_iqr_mask]))) if top_iqr_mask.any() else np.nan

    return metrics


def compute_metrics_for_indices(pred_minutes: np.ndarray, indices_cpu: np.ndarray) -> Dict[str, float]:
    """Compute metrics for selected original dataframe row indices."""
    y_true = y_np[indices_cpu]
    w = df.iloc[indices_cpu]["sample_weight"].to_numpy(dtype=np.float32)
    return compute_metrics_from_arrays(y_true, pred_minutes, row_weights=w)

## 15. Graph model definitions

Both models use the same residualized monotone quantile head.  If the decoder emits zero deltas, predictions equal the Cold-OD fallback prior.  This makes the graph model learn residual corrections rather than absolute quantiles from scratch.

In [15]:
class ResidualizedMonotoneHead(nn.Module):
    """Residualized monotone quantile head around a base prior."""

    def __init__(self, hidden_dim: int):
        super().__init__()
        self.out = nn.Linear(hidden_dim, 3)
        # Start exactly at the fallback prior when decoder hidden states are near zero.
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, hidden: torch.Tensor, base_scaled: torch.Tensor) -> torch.Tensor:
        """Return scaled q25/q50/q75 predictions with monotonicity guaranteed."""
        delta = self.out(hidden)

        base_q25 = torch.clamp(base_scaled[:, 0], min=1e-5)
        base_gap50 = torch.clamp(base_scaled[:, 1] - base_scaled[:, 0], min=1e-5)
        base_gap75 = torch.clamp(base_scaled[:, 2] - base_scaled[:, 1], min=1e-5)

        q25 = F.softplus(inv_softplus(base_q25) + delta[:, 0])
        gap50 = F.softplus(inv_softplus(base_gap50) + delta[:, 1])
        gap75 = F.softplus(inv_softplus(base_gap75) + delta[:, 2])
        q50 = q25 + gap50
        q75 = q50 + gap75
        return torch.stack([q25, q50, q75], dim=-1)


class EdgeDecoder(nn.Module):
    """Decode source/destination embeddings plus OD-year features into quantile residuals."""

    def __init__(self, hidden_dim: int, edge_dim: int, year_count: int, year_embedding_dim: int, edge_hidden_dim: int, dropout: float):
        super().__init__()
        self.year_embedding = nn.Embedding(year_count, year_embedding_dim)
        input_dim = hidden_dim * 4 + edge_dim + year_embedding_dim
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, edge_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(edge_hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.head = ResidualizedMonotoneHead(hidden_dim)

    def forward(self, src_h: torch.Tensor, dst_h: torch.Tensor, edge_attr_batch: torch.Tensor, base_scaled: torch.Tensor, year_idx_batch: torch.Tensor) -> torch.Tensor:
        """Decode edge-level predictions in scaled target units."""
        year_h = self.year_embedding(year_idx_batch)
        z = torch.cat([src_h, dst_h, torch.abs(src_h - dst_h), src_h * dst_h, edge_attr_batch, year_h], dim=-1)
        hidden = self.mlp(z)
        return self.head(hidden, base_scaled)


class GraphSAGEResidualQuantileModel(nn.Module):
    """Homogeneous GraphSAGE residual edge predictor."""

    def __init__(self, node_in_dim: int, edge_dim: int, year_count: int, config: ExperimentConfig):
        super().__init__()
        self.node_proj = nn.Linear(node_in_dim, config.hidden_dim)
        self.convs = nn.ModuleList([SAGEConv(config.hidden_dim, config.hidden_dim) for _ in range(config.graph_layers)])
        self.dropout = nn.Dropout(config.dropout)
        self.decoder = EdgeDecoder(
            hidden_dim=config.hidden_dim,
            edge_dim=edge_dim,
            year_count=year_count,
            year_embedding_dim=config.year_embedding_dim,
            edge_hidden_dim=config.edge_hidden_dim,
            dropout=config.dropout,
        )

    def encode(self, node_features: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """Encode FAF nodes with homogeneous GraphSAGE message passing."""
        h = F.gelu(self.node_proj(node_features))
        for conv in self.convs:
            h_next = conv(h, edge_index)
            h = F.gelu(h_next)
            h = self.dropout(h)
        return h

    def forward(self, node_features: torch.Tensor, edge_index: torch.Tensor, edge_label_index_batch: torch.Tensor, edge_attr_batch: torch.Tensor, base_scaled_batch: torch.Tensor, year_idx_batch: torch.Tensor) -> torch.Tensor:
        """Predict scaled quantiles for a batch of supervised OD-year edges."""
        h = self.encode(node_features, edge_index)
        src, dst = edge_label_index_batch
        return self.decoder(h[src], h[dst], edge_attr_batch, base_scaled_batch, year_idx_batch)


class HGTResidualQuantileModel(nn.Module):
    """Typed-relation HGT residual edge predictor."""

    def __init__(self, node_in_dim: int, edge_dim: int, year_count: int, metadata: Tuple[List[str], List[Tuple[str, str, str]]], config: ExperimentConfig):
        super().__init__()
        self.node_proj = nn.ModuleDict({"faf": nn.Linear(node_in_dim, config.hidden_dim)})
        self.convs = nn.ModuleList([
            HGTConv(config.hidden_dim, config.hidden_dim, metadata, heads=config.hgt_heads)
            for _ in range(config.graph_layers)
        ])
        self.dropout = nn.Dropout(config.dropout)
        self.decoder = EdgeDecoder(
            hidden_dim=config.hidden_dim,
            edge_dim=edge_dim,
            year_count=year_count,
            year_embedding_dim=config.year_embedding_dim,
            edge_hidden_dim=config.edge_hidden_dim,
            dropout=config.dropout,
        )

    def encode(self, x_dict: Mapping[str, torch.Tensor], edge_index_dict: Mapping[Tuple[str, str, str], torch.Tensor]) -> Dict[str, torch.Tensor]:
        """Encode FAF nodes with typed HGTConv message passing."""
        h_dict = {node_type: F.gelu(self.node_proj[node_type](x)) for node_type, x in x_dict.items()}
        for conv in self.convs:
            h_dict = conv(h_dict, edge_index_dict)
            h_dict = {node_type: self.dropout(F.gelu(h)) for node_type, h in h_dict.items()}
        return h_dict

    def forward(self, x_dict: Mapping[str, torch.Tensor], edge_index_dict: Mapping[Tuple[str, str, str], torch.Tensor], edge_label_index_batch: torch.Tensor, edge_attr_batch: torch.Tensor, base_scaled_batch: torch.Tensor, year_idx_batch: torch.Tensor) -> torch.Tensor:
        """Predict scaled quantiles for a batch of supervised OD-year edges."""
        h_dict = self.encode(x_dict, edge_index_dict)
        src, dst = edge_label_index_batch
        h = h_dict["faf"]
        return self.decoder(h[src], h[dst], edge_attr_batch, base_scaled_batch, year_idx_batch)

## 16. Training and prediction utilities

This version trains each model for five seeds.  During training, it tracks three validation criteria and saves one checkpoint per criterion:

- lowest validation `pinball_mean`
- lowest validation `mae_q75`
- lowest validation `iqr_mae`

Early stopping is still based on validation weighted pinball because that is the main weighted training objective.

In [16]:
def make_batches(indices: torch.Tensor, batch_size: int, shuffle: bool) -> Iterable[torch.Tensor]:
    """Yield mini-batches of row indices."""
    if shuffle:
        perm = indices[torch.randperm(indices.numel(), device=indices.device)]
    else:
        perm = indices
    for start in range(0, perm.numel(), batch_size):
        yield perm[start:start + batch_size]


def forward_model(model: nn.Module, model_type: str, indices: torch.Tensor) -> torch.Tensor:
    """Forward pass for a selected batch of original dataframe row indices."""
    if model_type == "graphsage":
        return model(
            node_features=node_x,
            edge_index=hom_edge_index,
            edge_label_index_batch=edge_label_index[:, indices],
            edge_attr_batch=edge_attr[indices],
            base_scaled_batch=base_prior_scaled[indices],
            year_idx_batch=year_idx[indices],
        )
    if model_type == "hgt":
        return model(
            x_dict={"faf": node_x},
            edge_index_dict=typed_edge_index_dict,
            edge_label_index_batch=edge_label_index[:, indices],
            edge_attr_batch=edge_attr[indices],
            base_scaled_batch=base_prior_scaled[indices],
            year_idx_batch=year_idx[indices],
        )
    raise ValueError(f"Unsupported model_type: {model_type}")


def predict_model(model: nn.Module, model_type: str, indices: torch.Tensor, batch_size: int = 16384) -> np.ndarray:
    """Predict q25/q50/q75 in minutes for selected rows."""
    model.eval()
    outputs: List[np.ndarray] = []
    with torch.no_grad():
        for batch in make_batches(indices, batch_size=batch_size, shuffle=False):
            pred_scaled = forward_model(model, model_type, batch)
            pred = pred_scaled.detach().cpu().numpy() * target_scale
            outputs.append(pred.astype(np.float32))
    return np.concatenate(outputs, axis=0) if outputs else np.zeros((0, 3), dtype=np.float32)


def evaluate_validation(model: nn.Module, model_type: str, indices: torch.Tensor) -> Dict[str, float]:
    """Evaluate validation/test metrics for early stopping and checkpoint selection."""
    pred = predict_model(model, model_type, indices)
    idx_np = indices.detach().cpu().numpy()
    return compute_metrics_for_indices(pred, idx_np)


def state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    """Copy a model state dict to CPU tensors for checkpoint storage."""
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


@dataclass
class SeedTrainResult:
    """Container for one model/seed training run."""

    model_name: str
    model_type: str
    seed: int
    model_factory_kwargs: Dict[str, Any]
    best_states: Dict[str, Dict[str, torch.Tensor]]
    best_records: Dict[str, Dict[str, float]]
    history: pd.DataFrame


def train_one_graph_model(
    model_name: str,
    model_type: str,
    model: nn.Module,
    seed: int,
    config: ExperimentConfig,
) -> SeedTrainResult:
    """Train one graph model for one seed and save three validation-selected states."""
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)

    best_weighted_pinball = math.inf
    epochs_without_improvement = 0

    # Three requested checkpoint selectors.
    best_values = {
        "best_val_pinball": math.inf,
        "best_val_q75": math.inf,
        "best_val_iqr": math.inf,
    }
    best_states: Dict[str, Dict[str, torch.Tensor]] = {}
    best_records: Dict[str, Dict[str, float]] = {}
    history_rows: List[Dict[str, Any]] = []

    start_time = time.time()
    for epoch in range(1, config.max_epochs + 1):
        model.train()
        batch_losses: List[float] = []
        for batch in make_batches(split_indices["train"], config.batch_size, shuffle=True):
            pred_scaled = forward_model(model, model_type, batch)
            loss = training_loss(pred_scaled, y_scaled[batch], weights[batch], config.lambda_iqr)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            if config.grad_clip_norm > 0:
                nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip_norm)
            optimizer.step()
            batch_losses.append(float(loss.detach().cpu()))

        if epoch % config.eval_every == 0:
            val_metrics = evaluate_validation(model, model_type, split_indices["val"])
            train_loss = float(np.mean(batch_losses)) if batch_losses else np.nan

            record = {
                "model": model_name,
                "model_type": model_type,
                "seed": seed,
                "epoch": epoch,
                "train_loss_scaled": train_loss,
                "elapsed_seconds": time.time() - start_time,
                **{f"val_{key}": value for key, value in val_metrics.items()},
            }
            history_rows.append(record)

            # Save requested checkpoints.
            selector_values = {
                "best_val_pinball": val_metrics["pinball_mean"],
                "best_val_q75": val_metrics["mae_q75"],
                "best_val_iqr": val_metrics["iqr_mae"],
            }
            for selector_name, selector_value in selector_values.items():
                if selector_value < best_values[selector_name]:
                    best_values[selector_name] = selector_value
                    best_states[selector_name] = state_dict_to_cpu(model)
                    best_records[selector_name] = {
                        "model": model_name,
                        "model_type": model_type,
                        "seed": seed,
                        "checkpoint_metric": selector_name,
                        "best_epoch": epoch,
                        "selector_value": float(selector_value),
                        "val_pinball_mean": float(val_metrics["pinball_mean"]),
                        "val_weighted_pinball_mean": float(val_metrics["weighted_pinball_mean"]),
                        "val_mae_q75": float(val_metrics["mae_q75"]),
                        "val_iqr_mae": float(val_metrics["iqr_mae"]),
                    }

            # Early stopping monitors weighted validation pinball.
            current_weighted = float(val_metrics["weighted_pinball_mean"])
            if current_weighted < best_weighted_pinball:
                best_weighted_pinball = current_weighted
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += config.eval_every

            if epoch % 25 == 0 or epoch == 1:
                print(
                    f"[{model_name} | seed={seed}] epoch={epoch:04d} "
                    f"train_loss={train_loss:.5f} "
                    f"val_pinball={val_metrics['pinball_mean']:.3f} "
                    f"val_q75={val_metrics['mae_q75']:.3f} "
                    f"val_iqr={val_metrics['iqr_mae']:.3f}"
                )

            if epochs_without_improvement >= config.patience:
                print(f"Early stopping {model_name} seed={seed} at epoch {epoch}.")
                break

    # Ensure all requested checkpoints exist even if validation did not run for some reason.
    for selector_name in CHECKPOINT_METRICS:
        if selector_name not in best_states:
            best_states[selector_name] = state_dict_to_cpu(model)
            best_records[selector_name] = {
                "model": model_name,
                "model_type": model_type,
                "seed": seed,
                "checkpoint_metric": selector_name,
                "best_epoch": epoch,
                "selector_value": np.nan,
            }

    history_df = pd.DataFrame(history_rows)
    return SeedTrainResult(
        model_name=model_name,
        model_type=model_type,
        seed=seed,
        model_factory_kwargs={},
        best_states=best_states,
        best_records=best_records,
        history=history_df,
    )

## 17. Train GraphSAGE and HGT over five seeds

This cell is the main training loop.  It trains GraphSAGE and HGT for the requested seeds and stores validation-selected checkpoint states.  Checkpoints are also written to disk so later notebooks can reload them without retraining.

In [17]:
train_results: List[SeedTrainResult] = []
checkpoint_records: List[Dict[str, Any]] = []
metadata = (["faf"], list(typed_edge_index_dict_cpu.keys()))
year_count = len(year_to_idx)

for seed in cfg.seeds:
    set_global_seed(seed)

    if cfg.run_graphsage:
        graphsage_model = GraphSAGEResidualQuantileModel(
            node_in_dim=node_x.shape[1],
            edge_dim=edge_attr.shape[1],
            year_count=year_count,
            config=cfg,
        )
        result = train_one_graph_model(
            model_name="graphsage_residual_prior_features",
            model_type="graphsage",
            model=graphsage_model,
            seed=seed,
            config=cfg,
        )
        train_results.append(result)

    if cfg.run_hgt:
        hgt_model = HGTResidualQuantileModel(
            node_in_dim=node_x.shape[1],
            edge_dim=edge_attr.shape[1],
            year_count=year_count,
            metadata=metadata,
            config=cfg,
        )
        result = train_one_graph_model(
            model_name="hgt_residual_prior_features",
            model_type="hgt",
            model=hgt_model,
            seed=seed,
            config=cfg,
        )
        train_results.append(result)

# Save checkpoint files and build checkpoint summary rows.
for result in train_results:
    seed_dir = models_dir / result.model_name / f"seed_{result.seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    for checkpoint_metric, state in result.best_states.items():
        record = result.best_records.get(checkpoint_metric, {}).copy()
        checkpoint_path = seed_dir / f"{checkpoint_metric}.pt"
        if cfg.save_model_checkpoints:
            torch.save(
                {
                    "state_dict": state,
                    "model_name": result.model_name,
                    "model_type": result.model_type,
                    "seed": result.seed,
                    "checkpoint_metric": checkpoint_metric,
                    "config": asdict(cfg),
                    "node_feature_columns": node_feature_columns,
                    "edge_feature_columns": edge_feature_columns,
                    "target_scale": target_scale,
                    "year_to_idx": year_to_idx,
                    "best_record": record,
                },
                checkpoint_path,
            )
        record["checkpoint_path"] = str(checkpoint_path)
        checkpoint_records.append(record)

training_history = pd.concat([result.history for result in train_results], ignore_index=True) if train_results else pd.DataFrame()
checkpoint_summary = pd.DataFrame(checkpoint_records)

print("Training history rows:", len(training_history))
print("Checkpoint summary:")
print(checkpoint_summary[["model", "seed", "checkpoint_metric", "best_epoch", "selector_value"]].head(20))

[graphsage_residual_prior_features | seed=7] epoch=0001 train_loss=0.20301 val_pinball=436.932 val_q75=1310.554 val_iqr=836.746
[graphsage_residual_prior_features | seed=7] epoch=0025 train_loss=0.06563 val_pinball=192.615 val_q75=734.916 val_iqr=607.196
[graphsage_residual_prior_features | seed=7] epoch=0050 train_loss=0.05042 val_pinball=178.265 val_q75=683.739 val_iqr=573.687
[graphsage_residual_prior_features | seed=7] epoch=0075 train_loss=0.04368 val_pinball=172.131 val_q75=643.139 val_iqr=540.485
[graphsage_residual_prior_features | seed=7] epoch=0100 train_loss=0.03976 val_pinball=173.183 val_q75=655.213 val_iqr=547.427
[graphsage_residual_prior_features | seed=7] epoch=0125 train_loss=0.03697 val_pinball=175.376 val_q75=652.043 val_iqr=540.794
[graphsage_residual_prior_features | seed=7] epoch=0150 train_loss=0.03561 val_pinball=172.547 val_q75=637.202 val_iqr=525.776
[graphsage_residual_prior_features | seed=7] epoch=0175 train_loss=0.03383 val_pinball=169.044 val_q75=627.728

## 18. Build graph prediction tables for every seed and checkpoint

Predictions are generated for validation and test rows for each saved checkpoint.  This creates a by-seed prediction table that can support both individual-seed reporting and seed-ensemble reporting.

In [18]:
def instantiate_model(model_name: str, model_type: str) -> nn.Module:
    """Instantiate a fresh graph model with the configured architecture."""
    if model_type == "graphsage":
        return GraphSAGEResidualQuantileModel(
            node_in_dim=node_x.shape[1],
            edge_dim=edge_attr.shape[1],
            year_count=year_count,
            config=cfg,
        )
    if model_type == "hgt":
        return HGTResidualQuantileModel(
            node_in_dim=node_x.shape[1],
            edge_dim=edge_attr.shape[1],
            year_count=year_count,
            metadata=metadata,
            config=cfg,
        )
    raise ValueError(f"Unsupported model_type: {model_type}")


def build_prediction_frame(
    model_name: str,
    model_type: str,
    seed: int,
    checkpoint_metric: str,
    state_dict: Mapping[str, torch.Tensor],
    splits: Sequence[str] = ("val", "test"),
) -> pd.DataFrame:
    """Build a normalized prediction dataframe for one model/seed/checkpoint."""
    model = instantiate_model(model_name, model_type).to(device)
    model.load_state_dict({key: value.to(device) for key, value in state_dict.items()})
    model.eval()

    frames: List[pd.DataFrame] = []
    for split_name in splits:
        indices = split_indices[split_name]
        idx_np = indices.detach().cpu().numpy()
        rows = df.iloc[idx_np].copy()
        pred = predict_model(model, model_type, indices)

        frame = pd.DataFrame({
            "source": "GRAPH",
            "model": model_name,
            "variant": "postprocessed",
            "checkpoint_metric": checkpoint_metric,
            "seed": seed,
            "split": split_name,
            origin_col: rows[origin_col].to_numpy(),
            dest_col: rows[dest_col].to_numpy(),
            year_col: rows[year_col].to_numpy(dtype=np.int64),
            "od_pair": rows["od_pair"].to_numpy(),
            "true_q25": rows["truck_q25"].to_numpy(dtype=np.float32),
            "true_q50": rows["truck_q50"].to_numpy(dtype=np.float32),
            "true_q75": rows["truck_q75"].to_numpy(dtype=np.float32),
            "pred_q25": pred[:, 0],
            "pred_q50": pred[:, 1],
            "pred_q75": pred[:, 2],
            "sample_weight": rows["sample_weight"].to_numpy(dtype=np.float32),
        })
        if "n_fmi_county_pairs" in rows.columns:
            frame["n_fmi_county_pairs"] = pd.to_numeric(rows["n_fmi_county_pairs"], errors="coerce").to_numpy(dtype=np.float32)
        else:
            frame["n_fmi_county_pairs"] = np.nan
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


graph_prediction_frames: List[pd.DataFrame] = []
for result in train_results:
    for checkpoint_metric, state in result.best_states.items():
        graph_prediction_frames.append(
            build_prediction_frame(
                model_name=result.model_name,
                model_type=result.model_type,
                seed=result.seed,
                checkpoint_metric=checkpoint_metric,
                state_dict=state,
            )
        )

graph_predictions_by_seed = pd.concat(graph_prediction_frames, ignore_index=True) if graph_prediction_frames else pd.DataFrame()
print("Graph by-seed prediction shape:", graph_predictions_by_seed.shape)
print(graph_predictions_by_seed[["source", "model", "seed", "checkpoint_metric", "split"]].drop_duplicates().head(20))

Graph by-seed prediction shape: (60420, 18)
      source                              model  seed checkpoint_metric split
0      GRAPH  graphsage_residual_prior_features     7  best_val_pinball   val
957    GRAPH  graphsage_residual_prior_features     7  best_val_pinball  test
2014   GRAPH  graphsage_residual_prior_features     7      best_val_q75   val
2971   GRAPH  graphsage_residual_prior_features     7      best_val_q75  test
4028   GRAPH  graphsage_residual_prior_features     7      best_val_iqr   val
4985   GRAPH  graphsage_residual_prior_features     7      best_val_iqr  test
6042   GRAPH        hgt_residual_prior_features     7  best_val_pinball   val
6999   GRAPH        hgt_residual_prior_features     7  best_val_pinball  test
8056   GRAPH        hgt_residual_prior_features     7      best_val_q75   val
9013   GRAPH        hgt_residual_prior_features     7      best_val_q75  test
10070  GRAPH        hgt_residual_prior_features     7      best_val_iqr   val
11027  GRAPH        

## 19. Error columns and seed-ensemble predictions

This cell attaches row-level error columns and builds seed-ensemble predictions by averaging q25/q50/q75 across the five seeds for each graph model and checkpoint metric.

In [19]:
def add_error_columns(pred_df: pd.DataFrame) -> pd.DataFrame:
    """Attach row-level errors, IQR errors, and pinball losses."""
    if pred_df.empty:
        return pred_df.copy()
    result = pred_df.copy()
    y_true = result[["true_q25", "true_q50", "true_q75"]].to_numpy(dtype=np.float32)
    y_pred = result[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(dtype=np.float32)

    result["error_q25"] = y_pred[:, 0] - y_true[:, 0]
    result["error_q50"] = y_pred[:, 1] - y_true[:, 1]
    result["error_q75"] = y_pred[:, 2] - y_true[:, 2]
    result["abs_error_q25"] = np.abs(result["error_q25"])
    result["abs_error_q50"] = np.abs(result["error_q50"])
    result["abs_error_q75"] = np.abs(result["error_q75"])
    result["true_iqr"] = y_true[:, 2] - y_true[:, 0]
    result["pred_iqr"] = y_pred[:, 2] - y_pred[:, 0]
    result["abs_error_iqr"] = np.abs(result["pred_iqr"] - result["true_iqr"])

    pinball = numpy_pinball(y_true, y_pred, TAU_VALUES)
    result["pinball_q25"] = pinball[:, 0]
    result["pinball_q50"] = pinball[:, 1]
    result["pinball_q75"] = pinball[:, 2]
    result["pinball_mean_row"] = pinball.mean(axis=1)
    return result


def build_seed_ensemble_predictions(pred_df: pd.DataFrame) -> pd.DataFrame:
    """Average graph predictions across seeds for each model/checkpoint/split/key."""
    if pred_df.empty:
        return pred_df.copy()

    key_cols = ["source", "model", "variant", "checkpoint_metric", "split", origin_col, dest_col, year_col, "od_pair"]
    truth_cols = ["true_q25", "true_q50", "true_q75", "sample_weight", "n_fmi_county_pairs"]

    grouped_pred = pred_df.groupby(key_cols, as_index=False)[["pred_q25", "pred_q50", "pred_q75"]].mean()
    grouped_truth = pred_df.groupby(key_cols, as_index=False)[truth_cols].first()
    ensemble = grouped_pred.merge(grouped_truth, on=key_cols, how="left")
    ensemble["source"] = "GRAPH_ENSEMBLE"
    ensemble["seed"] = "mean"
    return ensemble


graph_predictions_by_seed = add_error_columns(graph_predictions_by_seed)
graph_predictions_seed_ensemble = build_seed_ensemble_predictions(graph_predictions_by_seed) if cfg.build_seed_ensembles else pd.DataFrame()
graph_predictions_seed_ensemble = add_error_columns(graph_predictions_seed_ensemble)

print("Seed-ensemble prediction shape:", graph_predictions_seed_ensemble.shape)
print(graph_predictions_seed_ensemble[["source", "model", "checkpoint_metric", "split"]].drop_duplicates().head(20))

Seed-ensemble prediction shape: (12084, 31)
               source                              model checkpoint_metric  \
0      GRAPH_ENSEMBLE  graphsage_residual_prior_features      best_val_iqr   
1057   GRAPH_ENSEMBLE  graphsage_residual_prior_features      best_val_iqr   
2014   GRAPH_ENSEMBLE  graphsage_residual_prior_features  best_val_pinball   
3071   GRAPH_ENSEMBLE  graphsage_residual_prior_features  best_val_pinball   
4028   GRAPH_ENSEMBLE  graphsage_residual_prior_features      best_val_q75   
5085   GRAPH_ENSEMBLE  graphsage_residual_prior_features      best_val_q75   
6042   GRAPH_ENSEMBLE        hgt_residual_prior_features      best_val_iqr   
7099   GRAPH_ENSEMBLE        hgt_residual_prior_features      best_val_iqr   
8056   GRAPH_ENSEMBLE        hgt_residual_prior_features  best_val_pinball   
9113   GRAPH_ENSEMBLE        hgt_residual_prior_features  best_val_pinball   
10070  GRAPH_ENSEMBLE        hgt_residual_prior_features      best_val_q75   
11127  GRAPH_ENSEMBL

## 20. Load previous Cold-OD non-graph baseline predictions

The graph results are compared against the previous Cold-OD non-graph baselines.  The schema is normalized so metrics can be recomputed directly from predictions instead of relying on potentially stale CSV artifacts.

In [20]:
def normalize_baseline_prediction_schema(
    baseline_df: pd.DataFrame,
    origin_col_target: str,
    dest_col_target: str,
    year_col_target: str,
) -> pd.DataFrame:
    """Normalize non-graph baseline predictions into the graph-analysis schema."""
    result = normalize_prediction_key_columns(baseline_df, origin_col_target, dest_col_target, year_col_target)
    result = result.copy()

    if "source" not in result.columns:
        result["source"] = "ColdOD"
    result["source"] = result["source"].replace({"NON_GRAPH": "ColdOD", "BASELINE": "ColdOD"})
    if "variant" not in result.columns:
        result["variant"] = "postprocessed"
    if "checkpoint_metric" not in result.columns:
        result["checkpoint_metric"] = "not_applicable"
    if "seed" not in result.columns:
        result["seed"] = "not_applicable"

    mapping_candidates = {
        "true_q25": ["true_q25", "truck_q25", "y_q25"],
        "true_q50": ["true_q50", "truck_q50", "y_q50"],
        "true_q75": ["true_q75", "truck_q75", "y_q75"],
        "pred_q25": ["pred_q25", "q25_pred", "hat_q25", "prediction_q25"],
        "pred_q50": ["pred_q50", "q50_pred", "hat_q50", "prediction_q50"],
        "pred_q75": ["pred_q75", "q75_pred", "hat_q75", "prediction_q75"],
    }
    for target_col, candidates in mapping_candidates.items():
        if target_col not in result.columns:
            source_col = find_first_existing_column(result, candidates, target_col)
            result[target_col] = result[source_col]
        result[target_col] = pd.to_numeric(result[target_col], errors="coerce").astype(np.float32)

    if "sample_weight" not in result.columns:
        keys = result[[origin_col_target, dest_col_target, year_col_target]].copy()
        weights_lookup = df[[origin_col_target, dest_col_target, year_col_target, "sample_weight"]].copy()
        result = result.merge(weights_lookup, on=[origin_col_target, dest_col_target, year_col_target], how="left")
    result["sample_weight"] = pd.to_numeric(result["sample_weight"], errors="coerce").fillna(1.0).astype(np.float32)

    if "n_fmi_county_pairs" not in result.columns:
        if "n_fmi_county_pairs" in df.columns:
            n_lookup = df[[origin_col_target, dest_col_target, year_col_target, "n_fmi_county_pairs"]].copy()
            result = result.merge(n_lookup, on=[origin_col_target, dest_col_target, year_col_target], how="left")
        else:
            result["n_fmi_county_pairs"] = np.nan
    result["n_fmi_county_pairs"] = pd.to_numeric(result["n_fmi_county_pairs"], errors="coerce")
    result["od_pair"] = result[origin_col_target].astype(str) + "->" + result[dest_col_target].astype(str)

    keep_cols = [
        "source", "model", "variant", "checkpoint_metric", "seed", "split",
        origin_col_target, dest_col_target, year_col_target, "od_pair",
        "true_q25", "true_q50", "true_q75", "pred_q25", "pred_q50", "pred_q75",
        "sample_weight", "n_fmi_county_pairs",
    ]
    return result[keep_cols].copy()


if cold_baseline_predictions_path.exists():
    baseline_raw = pd.read_parquet(cold_baseline_predictions_path)
    baseline_predictions = normalize_baseline_prediction_schema(baseline_raw, origin_col, dest_col, year_col)
    baseline_predictions = add_error_columns(baseline_predictions)
    print("Loaded Cold-OD non-graph predictions:", baseline_predictions.shape)
    print(baseline_predictions[["source", "model", "split"]].drop_duplicates().head(20))
else:
    baseline_predictions = pd.DataFrame()
    print("Cold-OD non-graph prediction file was not found. Combined comparisons will include graph models only.")

Loaded Cold-OD non-graph predictions: (22154, 31)
       source                                model split
0      ColdOD                  global_train_median   val
957    ColdOD                  global_train_median  test
2014   ColdOD                         origin_prior   val
2971   ColdOD                         origin_prior  test
4028   ColdOD                    destination_prior   val
4985   ColdOD                    destination_prior  test
6042   ColdOD       origin_destination_blend_prior   val
6999   ColdOD       origin_destination_blend_prior  test
8056   ColdOD     lightgbm_direct_current_features   val
9013   ColdOD     lightgbm_direct_current_features  test
10070  ColdOD            prior_feature_lgbm_direct   val
11027  ColdOD            prior_feature_lgbm_direct  test
12084  ColdOD       residual_lgbm_current_features   val
13041  ColdOD       residual_lgbm_current_features  test
14098  ColdOD         residual_lgbm_prior_features   val
15055  ColdOD         residual_lgbm_pr

## 21. Compute metrics, leaderboards, and multi-seed summaries

This cell recomputes metrics from prediction tables.  It produces:

- per-seed graph metrics,
- seed-ensemble graph metrics,
- combined graph/non-graph metrics,
- seed mean/std summaries for robustness.

In [21]:
def compute_metrics_for_prediction_frame(pred_df: pd.DataFrame) -> pd.DataFrame:
    """Compute metrics for each source/model/checkpoint/seed/split group."""
    if pred_df.empty:
        return pd.DataFrame()
    rows: List[Dict[str, Any]] = []
    group_cols = ["source", "model", "variant", "checkpoint_metric", "seed", "split"]
    for group_key, group in pred_df.groupby(group_cols, dropna=False):
        y_true = group[["true_q25", "true_q50", "true_q75"]].to_numpy(dtype=np.float32)
        y_pred = group[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(dtype=np.float32)
        w = group["sample_weight"].to_numpy(dtype=np.float32) if "sample_weight" in group.columns else None
        metrics = compute_metrics_from_arrays(y_true, y_pred, row_weights=w)
        row = dict(zip(group_cols, group_key))
        row.update(metrics)
        rows.append(row)
    return pd.DataFrame(rows)


graph_metrics_by_seed = compute_metrics_for_prediction_frame(graph_predictions_by_seed)
graph_metrics_seed_ensemble = compute_metrics_for_prediction_frame(graph_predictions_seed_ensemble)
baseline_metrics = compute_metrics_for_prediction_frame(baseline_predictions)

combined_predictions = pd.concat(
    [baseline_predictions, graph_predictions_seed_ensemble, graph_predictions_by_seed],
    ignore_index=True,
    sort=False,
)
combined_predictions = add_error_columns(combined_predictions)
combined_metrics = compute_metrics_for_prediction_frame(combined_predictions)

leaderboard_test = combined_metrics[combined_metrics["split"].eq("test")].copy()
leaderboard_test = leaderboard_test.sort_values(["pinball_mean", "mae_q75"], ascending=[True, True]).reset_index(drop=True)
leaderboard_test.insert(0, "rank", np.arange(1, len(leaderboard_test) + 1))

# Seed robustness summary for individual graph seeds only.
seed_summary_rows: List[Dict[str, Any]] = []
graph_test_seed_metrics = graph_metrics_by_seed[graph_metrics_by_seed["split"].eq("test")].copy()
for group_key, group in graph_test_seed_metrics.groupby(["source", "model", "checkpoint_metric"], dropna=False):
    row = dict(zip(["source", "model", "checkpoint_metric"], group_key))
    row["n_seeds"] = int(group["seed"].nunique())
    for metric in ["pinball_mean", "weighted_pinball_mean", "mae_q50", "mae_q75", "iqr_mae", "stress_top10_mae_q75", "top_iqr10_iqr_mae"]:
        if metric in group.columns:
            row[f"{metric}_mean"] = float(group[metric].mean())
            row[f"{metric}_std"] = float(group[metric].std(ddof=1)) if len(group) > 1 else 0.0
    seed_summary_rows.append(row)
seed_summary = pd.DataFrame(seed_summary_rows).sort_values("pinball_mean_mean", ascending=True).reset_index(drop=True)

print("Top test leaderboard rows:")
print(leaderboard_test[["rank", "source", "model", "checkpoint_metric", "seed", "pinball_mean", "mae_q75", "iqr_mae"]].head(20))
print("\nSeed summary:")
print(seed_summary.head(20))

Top test leaderboard rows:
    rank          source                              model checkpoint_metric  \
0      1  GRAPH_ENSEMBLE        hgt_residual_prior_features      best_val_iqr   
1      2  GRAPH_ENSEMBLE        hgt_residual_prior_features  best_val_pinball   
2      3  GRAPH_ENSEMBLE        hgt_residual_prior_features      best_val_q75   
3      4  GRAPH_ENSEMBLE  graphsage_residual_prior_features      best_val_iqr   
4      5  GRAPH_ENSEMBLE  graphsage_residual_prior_features  best_val_pinball   
5      6  GRAPH_ENSEMBLE  graphsage_residual_prior_features      best_val_q75   
6      7           GRAPH        hgt_residual_prior_features      best_val_iqr   
7      8           GRAPH        hgt_residual_prior_features      best_val_q75   
8      9           GRAPH        hgt_residual_prior_features  best_val_pinball   
9     10           GRAPH  graphsage_residual_prior_features  best_val_pinball   
10    11           GRAPH  graphsage_residual_prior_features      best_val_iqr   
1

## 22. Strict test-only segment summaries

This is one of the main v2 corrections.  Segment summaries are computed only from `split == "test"`.  The cell also asserts that each prediction identity contributes exactly the same number of test rows as the Cold-OD test set.

In [22]:
def assign_decile(series: pd.Series, name: str) -> pd.Series:
    """Assign robust decile labels with duplicate-edge handling."""
    try:
        bins = pd.qcut(series, q=10, duplicates="drop")
    except ValueError:
        bins = pd.qcut(series.rank(method="average"), q=10, duplicates="drop")
    return bins.astype(str).radd(f"{name}_")


def assign_quartile(series: pd.Series, name: str) -> pd.Series:
    """Assign robust quartile labels with duplicate-edge handling."""
    try:
        bins = pd.qcut(series, q=4, duplicates="drop")
    except ValueError:
        bins = pd.qcut(series.rank(method="average"), q=4, duplicates="drop")
    return bins.astype(str).radd(f"{name}_")


def add_test_segments(pred_df: pd.DataFrame) -> pd.DataFrame:
    """Add test-only segment labels using true values from test rows."""
    test_df = pred_df[pred_df["split"].eq("test")].copy()
    if test_df.empty:
        return test_df

    # Segment labels are based on unique OD-year test rows, then merged back to all model predictions.
    key_cols = [origin_col, dest_col, year_col]
    unique_test = test_df[key_cols + ["true_q75", "true_iqr", "n_fmi_county_pairs"]].drop_duplicates(key_cols).copy()
    unique_test["true_q75_decile"] = assign_decile(unique_test["true_q75"], "true_q75_decile")
    unique_test["true_iqr_decile"] = assign_decile(unique_test["true_iqr"], "true_iqr_decile")
    if unique_test["n_fmi_county_pairs"].notna().any():
        unique_test["n_fmi_county_pairs_quartile"] = assign_quartile(unique_test["n_fmi_county_pairs"].fillna(unique_test["n_fmi_county_pairs"].median()), "n_fmi_county_pairs_quartile")
    else:
        unique_test["n_fmi_county_pairs_quartile"] = "n_fmi_county_pairs_quartile_missing"

    segmented = test_df.merge(unique_test[key_cols + ["true_q75_decile", "true_iqr_decile", "n_fmi_county_pairs_quartile"]], on=key_cols, how="left")
    return segmented


def segment_summary_test_only(pred_df: pd.DataFrame, segment_col: str) -> pd.DataFrame:
    """Compute segment metrics strictly on test rows."""
    test_df = add_test_segments(pred_df)
    if test_df.empty or segment_col not in test_df.columns:
        return pd.DataFrame()

    group_cols = ["source", "model", "variant", "checkpoint_metric", "seed", segment_col]
    rows: List[Dict[str, Any]] = []
    for group_key, group in test_df.groupby(group_cols, dropna=False):
        y_true = group[["true_q25", "true_q50", "true_q75"]].to_numpy(dtype=np.float32)
        y_pred = group[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(dtype=np.float32)
        w = group["sample_weight"].to_numpy(dtype=np.float32)
        row = dict(zip(group_cols, group_key))
        row.update(compute_metrics_from_arrays(y_true, y_pred, row_weights=w))
        rows.append(row)
    summary = pd.DataFrame(rows)

    # Strict assertion: every model/checkpoint/seed identity must sum to the number of test rows.
    expected_n = int(df["cold_split"].eq("test").sum())
    identity_cols = ["source", "model", "variant", "checkpoint_metric", "seed"]
    totals = summary.groupby(identity_cols, dropna=False)["n_rows"].sum()
    bad = totals[~np.isclose(totals, expected_n)]
    if not bad.empty:
        raise AssertionError(
            f"Segment summary is not strictly test-only or has duplicate/missing test rows. Expected {expected_n}; bad totals: {bad.head().to_dict()}"
        )
    return summary.sort_values(identity_cols + [segment_col]).reset_index(drop=True)


segment_tables: Dict[str, pd.DataFrame] = {}
for segment_col in ["true_q75_decile", "true_iqr_decile", "n_fmi_county_pairs_quartile"]:
    segment_tables[segment_col] = segment_summary_test_only(combined_predictions, segment_col)
    print(segment_col, "rows:", len(segment_tables[segment_col]))
    if not segment_tables[segment_col].empty:
        print(segment_tables[segment_col].head(3))

true_q75_decile rows: 470
   source              model        variant checkpoint_metric            seed  \
0  ColdOD  destination_prior  postprocessed    not_applicable  not_applicable   
1  ColdOD  destination_prior  postprocessed    not_applicable  not_applicable   
2  ColdOD  destination_prior  postprocessed    not_applicable  not_applicable   

                                  true_q75_decile  n_rows      mae_q25  \
0  true_q75_decile_(127.29899999999999, 1510.928]   106.0  1329.721313   
1             true_q75_decile_(1510.928, 2182.09]   106.0   887.165894   
2              true_q75_decile_(2182.09, 2814.51]   105.0   557.630676   

       mae_q50      mae_q75  ...  raw_crossing_rate  \
0  1835.338135  2576.372070  ...                0.0   
1  1061.732422  1745.496338  ...                0.0   
2   713.748230   975.334900  ...                0.0   

   q50_inside_pred_q25_q75_rate  weighted_pinball_mean  \
0                      0.000000             850.081665   
1              

## 23. Strict test-only paired comparisons

This is the second major v2 correction.  Paired comparisons are computed only from `split == "test"`.  The default reference is the strongest Cold-OD non-graph baseline from the previous stage: `ColdOD::mlp_residual_prior_features`.

Positive deltas mean the graph model is better than the reference baseline.

In [23]:
def prediction_identity(frame: pd.DataFrame) -> pd.Series:
    """Create a stable model identity string including source, checkpoint, and seed."""
    return (
        frame["source"].astype(str)
        + "::" + frame["model"].astype(str)
        + "::" + frame["checkpoint_metric"].astype(str)
        + "::seed=" + frame["seed"].astype(str)
    )


def make_paired_comparisons_test_only(
    pred_df: pd.DataFrame,
    reference_source: str = "ColdOD",
    reference_model: str = "mlp_residual_prior_features",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Compute strict test-only paired comparisons between graph predictions and a reference baseline."""
    test_df = pred_df[pred_df["split"].eq("test")].copy()
    if test_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    test_df["identity"] = prediction_identity(test_df)
    key_cols = [origin_col, dest_col, year_col]
    reference_candidates = test_df[test_df["source"].astype(str).eq(reference_source) & test_df["model"].astype(str).eq(reference_model)].copy()
    if reference_candidates.empty:
        print(f"Reference baseline {reference_source}::{reference_model} not found; paired comparison skipped.")
        return pd.DataFrame(), pd.DataFrame()

    # If the baseline has multiple variants, use the first unique identity after sorting.
    reference_identity = sorted(reference_candidates["identity"].unique().tolist())[0]
    reference = reference_candidates[reference_candidates["identity"].eq(reference_identity)].copy()
    expected_n = int(df["cold_split"].eq("test").sum())
    if len(reference) != expected_n:
        raise AssertionError(f"Reference paired set should have {expected_n} rows, got {len(reference)}.")

    graph_part = test_df[test_df["source"].isin(["GRAPH", "GRAPH_ENSEMBLE"])].copy()
    ref_cols = key_cols + ["pinball_mean_row", "abs_error_q25", "abs_error_q50", "abs_error_q75", "abs_error_iqr"]
    reference_small = reference[ref_cols].copy()

    summary_rows: List[Dict[str, Any]] = []
    paired_frames: List[pd.DataFrame] = []
    metric_cols = ["pinball_mean_row", "abs_error_q25", "abs_error_q50", "abs_error_q75", "abs_error_iqr"]

    for candidate_identity, candidate in graph_part.groupby("identity"):
        merged = reference_small.merge(
            candidate[key_cols + metric_cols + ["source", "model", "checkpoint_metric", "seed"]],
            on=key_cols,
            suffixes=("_reference", "_candidate"),
            how="inner",
        )
        if len(merged) != expected_n:
            raise AssertionError(f"Candidate {candidate_identity} paired rows should be {expected_n}, got {len(merged)}.")

        for metric in metric_cols:
            merged[f"delta_{metric}"] = merged[f"{metric}_reference"] - merged[f"{metric}_candidate"]
            merged[f"candidate_wins_{metric}"] = merged[f"delta_{metric}"] > 0

        candidate_info = candidate[["source", "model", "checkpoint_metric", "seed"]].iloc[0].to_dict()
        row = {
            "reference_identity": reference_identity,
            "candidate_identity": candidate_identity,
            "n_rows": len(merged),
            **{f"candidate_{k}": v for k, v in candidate_info.items()},
        }
        for metric in metric_cols:
            row[f"mean_delta_{metric}"] = float(merged[f"delta_{metric}"].mean())
            row[f"median_delta_{metric}"] = float(merged[f"delta_{metric}"].median())
            row[f"win_rate_{metric}"] = float(merged[f"candidate_wins_{metric}"].mean())
        summary_rows.append(row)
        merged["reference_identity"] = reference_identity
        merged["candidate_identity"] = candidate_identity
        paired_frames.append(merged)

    summary = pd.DataFrame(summary_rows).sort_values("mean_delta_pinball_mean_row", ascending=False).reset_index(drop=True)
    paired_rows = pd.concat(paired_frames, ignore_index=True) if paired_frames else pd.DataFrame()
    return summary, paired_rows


paired_summary_test_only, paired_rows_test_only = make_paired_comparisons_test_only(combined_predictions)
print("Paired summary rows:", len(paired_summary_test_only))
print(paired_summary_test_only.head(20))

Paired summary rows: 36
                                   reference_identity  \
0   ColdOD::mlp_residual_prior_features::not_appli...   
1   ColdOD::mlp_residual_prior_features::not_appli...   
2   ColdOD::mlp_residual_prior_features::not_appli...   
3   ColdOD::mlp_residual_prior_features::not_appli...   
4   ColdOD::mlp_residual_prior_features::not_appli...   
5   ColdOD::mlp_residual_prior_features::not_appli...   
6   ColdOD::mlp_residual_prior_features::not_appli...   
7   ColdOD::mlp_residual_prior_features::not_appli...   
8   ColdOD::mlp_residual_prior_features::not_appli...   
9   ColdOD::mlp_residual_prior_features::not_appli...   
10  ColdOD::mlp_residual_prior_features::not_appli...   
11  ColdOD::mlp_residual_prior_features::not_appli...   
12  ColdOD::mlp_residual_prior_features::not_appli...   
13  ColdOD::mlp_residual_prior_features::not_appli...   
14  ColdOD::mlp_residual_prior_features::not_appli...   
15  ColdOD::mlp_residual_prior_features::not_appli...   
16  Col

## 24. Optional diagnostic plots

The plots are generated from strict test-only summaries and the recomputed leaderboard.  The notebook uses default Matplotlib styling.

In [24]:
if cfg.make_plots:
    try:
        import matplotlib.pyplot as plt

        # Prefer seed-ensemble rows for plots, while keeping all metrics in the CSV outputs.
        plot_leaderboard = leaderboard_test[
            leaderboard_test["source"].isin(["ColdOD", "GRAPH_ENSEMBLE"])
        ].copy()
        if plot_leaderboard.empty:
            plot_leaderboard = leaderboard_test.copy()
        plot_leaderboard["label"] = plot_leaderboard["source"].astype(str) + "::" + plot_leaderboard["model"].astype(str) + "::" + plot_leaderboard["checkpoint_metric"].astype(str)

        fig_df = plot_leaderboard.sort_values("pinball_mean", ascending=True).copy()
        plt.figure(figsize=(12, max(5, 0.42 * len(fig_df))))
        plt.barh(fig_df["label"], fig_df["pinball_mean"])
        plt.gca().invert_yaxis()
        plt.xlabel("pinball_mean")
        plt.title("Cold-OD Test Pinball Mean Leaderboard v2")
        plt.tight_layout()
        plt.savefig(plots_dir / "cold_od_graph_v2_leaderboard_pinball_mean.png", dpi=160)
        plt.close()

        fig_df = plot_leaderboard.sort_values("mae_q75", ascending=True).copy()
        plt.figure(figsize=(12, max(5, 0.42 * len(fig_df))))
        plt.barh(fig_df["label"], fig_df["mae_q75"])
        plt.gca().invert_yaxis()
        plt.xlabel("mae_q75")
        plt.title("Cold-OD Test q75 MAE Leaderboard v2")
        plt.tight_layout()
        plt.savefig(plots_dir / "cold_od_graph_v2_leaderboard_mae_q75.png", dpi=160)
        plt.close()

        # Test-only q75 decile plot for selected seed-ensemble graph and key non-graph baselines.
        q75_seg = segment_tables.get("true_q75_decile", pd.DataFrame()).copy()
        if not q75_seg.empty:
            selected_models = [
                ("GRAPH_ENSEMBLE", "hgt_residual_prior_features"),
                ("GRAPH_ENSEMBLE", "graphsage_residual_prior_features"),
                ("ColdOD", "mlp_residual_prior_features"),
                ("ColdOD", "prior_feature_lgbm_direct"),
                ("ColdOD", "residual_lgbm_prior_features"),
            ]
            plt.figure(figsize=(14, 6))
            for source_name, model_name in selected_models:
                part = q75_seg[(q75_seg["source"].eq(source_name)) & (q75_seg["model"].eq(model_name))]
                # Keep the best-val-pinball checkpoint for graph seed ensembles to avoid overplotting.
                if source_name == "GRAPH_ENSEMBLE":
                    part = part[part["checkpoint_metric"].eq("best_val_pinball")]
                if part.empty:
                    continue
                part = part.sort_values("true_q75_decile")
                label = f"{source_name}::{model_name}"
                plt.plot(part["true_q75_decile"], part["mae_q75"], marker="o", label=label)
            plt.xticks(rotation=45, ha="right")
            plt.ylabel("mae_q75")
            plt.xlabel("true_q75_decile")
            plt.title("Cold-OD Test q75 MAE by True q75 Decile v2")
            plt.legend()
            plt.tight_layout()
            plt.savefig(plots_dir / "cold_od_graph_v2_q75_mae_by_true_q75_decile.png", dpi=160)
            plt.close()

        print("Plots written to:", plots_dir)
    except Exception as exc:
        print("Plot generation skipped due to error:", repr(exc))

Plots written to: E:\NetworkOptimization\pythonProject1\Data\10_experiments\graphsage_hgt_cold_od_baselines_v2_notebook\east_plus_gulf\plots


## 25. Save v2 artifacts

This cell writes all v2 outputs with explicit file names so they cannot be confused with v1 outputs or non-graph Cold-OD outputs.

In [27]:
# ---------------------------------------------------------------------
# Save all v2 artifacts with Parquet-safe schema normalization
# ---------------------------------------------------------------------
#
# Why this cell is needed:
# Some combined prediction tables contain mixed-type identifier columns.
# For example, graph rows may store seed as an integer, while non-graph
# baseline rows may store seed as missing values or strings. Pandas then
# represents the column as object dtype, and pyarrow can fail with:
#
#   ArrowTypeError: Expected bytes, got a 'int' object
#
# The helper functions below normalize all object-like identifier columns
# before writing Parquet files. Numeric prediction and metric columns are
# preserved as numeric columns.

import json
import os
from pathlib import Path
from dataclasses import asdict

import numpy as np
import pandas as pd


# Ensure artifact directories exist.
output_dir = Path(output_dir)
tables_dir = Path(tables_dir)
plots_dir = Path(plots_dir)
models_dir = Path(models_dir)

for directory in [output_dir, tables_dir, plots_dir, models_dir]:
    directory.mkdir(parents=True, exist_ok=True)


# Columns that should always be stored as strings in prediction parquet files.
# This avoids mixed int/string object columns after concatenating graph and
# non-graph baseline predictions.
FORCE_STRING_COLUMNS = {
    "source",
    "model",
    "variant",
    "checkpoint",
    "seed",
    "split",
    "faf_orig",
    "faf_dest",
    "origin",
    "destination",
    "od_pair",
    "comparison",
    "reference_model",
    "candidate_model",
}


def json_default(value):
    """Convert NumPy, pandas, and Path objects into JSON-serializable objects."""
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.ndarray,)):
        return value.tolist()
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value) if not isinstance(value, (list, tuple, dict, set)) else False:
        return None
    return str(value)


def make_unique_columns(columns):
    """Return unique string column names while preserving the original order."""
    seen = {}
    unique_columns = []

    for column in columns:
        base_name = str(column)
        count = seen.get(base_name, 0)

        if count == 0:
            unique_columns.append(base_name)
        else:
            unique_columns.append(f"{base_name}__dup{count}")

        seen[base_name] = count + 1

    return unique_columns


def stringify_object_value(value):
    """Convert a scalar object value into a Parquet-safe string or missing value."""
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, sort_keys=True, default=json_default)

    if value is None:
        return pd.NA

    try:
        if pd.isna(value):
            return pd.NA
    except (TypeError, ValueError):
        pass

    return str(value)


def normalize_dataframe_for_parquet(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a copy of df with Parquet-safe dtypes.

    Rules:
    - Preserve numeric columns as numeric.
    - Preserve boolean columns as boolean where possible.
    - Convert object/string/category identifier columns to pandas string dtype.
    - Convert all remaining object/string/category columns to string dtype.
    - Ensure column names are unique strings.
    """
    clean = df.copy()
    clean.columns = make_unique_columns(clean.columns)

    for column in clean.columns:
        series = clean[column]
        dtype_name = str(series.dtype)

        should_force_string = column in FORCE_STRING_COLUMNS
        is_object_like = (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
            or dtype_name == "category"
        )

        if should_force_string or is_object_like:
            clean[column] = series.map(stringify_object_value).astype("string")

    return clean


def safe_to_parquet(df: pd.DataFrame, path: Path) -> Path:
    """
    Write a DataFrame to Parquet after dtype normalization.

    This function specifically prevents pyarrow mixed-object failures such as
    mixed integer/string values in the seed column.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    clean = normalize_dataframe_for_parquet(df)
    clean.to_parquet(path, index=False, engine="pyarrow")

    return path


def write_json(payload: dict, path: Path) -> Path:
    """Write a JSON artifact with robust serialization of NumPy/Pandas objects."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, default=json_default)

    return path


# ---------------------------------------------------------------------
# Define output paths.
# ---------------------------------------------------------------------

# Core prediction outputs.
graph_by_seed_path = output_dir / "predictions_graph_cold_od_val_test_by_seed_v2.parquet"
graph_seed_ensemble_path = output_dir / "predictions_graph_cold_od_val_test_seed_ensemble_v2.parquet"
combined_predictions_path = output_dir / "combined_predictions_graph_cold_od_val_test_v2.parquet"

# Core metric outputs.
metrics_by_seed_path = output_dir / "metrics_graph_cold_od_by_seed_v2.csv"
metrics_seed_ensemble_path = output_dir / "metrics_graph_cold_od_seed_ensemble_v2.csv"
combined_metrics_path = output_dir / "metrics_graph_cold_od_combined_v2.csv"
leaderboard_path = output_dir / "leaderboard_test_graph_cold_od_v2.csv"
seed_summary_path = output_dir / "seed_summary_graph_cold_od_v2.csv"
checkpoint_summary_path = output_dir / "checkpoint_summary_graph_cold_od_v2.csv"
history_path = output_dir / "training_history_graph_cold_od_v2.csv"

# Strict test-only outputs.
paired_summary_path = output_dir / "paired_summary_graph_vs_baseline_test_only_v2.csv"
paired_rows_path = output_dir / "paired_rows_graph_vs_baseline_test_only_v2.parquet"


# ---------------------------------------------------------------------
# Save prediction tables.
# ---------------------------------------------------------------------

safe_to_parquet(graph_predictions_by_seed, graph_by_seed_path)
safe_to_parquet(graph_predictions_seed_ensemble, graph_seed_ensemble_path)
safe_to_parquet(combined_predictions, combined_predictions_path)


# ---------------------------------------------------------------------
# Save metrics, histories, and summaries.
# ---------------------------------------------------------------------

graph_metrics_by_seed.to_csv(metrics_by_seed_path, index=False)
graph_metrics_seed_ensemble.to_csv(metrics_seed_ensemble_path, index=False)
combined_metrics.to_csv(combined_metrics_path, index=False)
leaderboard_test.to_csv(leaderboard_path, index=False)
seed_summary.to_csv(seed_summary_path, index=False)
checkpoint_summary.to_csv(checkpoint_summary_path, index=False)
training_history.to_csv(history_path, index=False)


# ---------------------------------------------------------------------
# Save strict test-only diagnostics.
# ---------------------------------------------------------------------

paired_summary_test_only.to_csv(paired_summary_path, index=False)

paired_rows_saved_path = None
if not paired_rows_test_only.empty:
    safe_to_parquet(paired_rows_test_only, paired_rows_path)
    paired_rows_saved_path = str(paired_rows_path)
else:
    # Remove stale paired-row parquet if the current run produced no paired rows.
    if paired_rows_path.exists():
        paired_rows_path.unlink()


# ---------------------------------------------------------------------
# Save test-only segment summaries.
# ---------------------------------------------------------------------

for segment_col, table in segment_tables.items():
    table_path = tables_dir / f"graph_v2_test_only_segment_summary__{segment_col}.csv"
    table.to_csv(table_path, index=False)

split_summary.to_csv(tables_dir / "cold_od_split_summary_v2.csv", index=False)
node_feature_df.to_csv(tables_dir / "faf_node_features_train_only_v2.csv", index=False)


# ---------------------------------------------------------------------
# Save feature and preprocessing metadata.
# ---------------------------------------------------------------------

feature_columns_payload = {
    "manifest_feature_columns": list(manifest_feature_columns),
    "cold_prior_feature_columns": list(COLD_PRIOR_COLUMNS),
    "edge_feature_columns": list(edge_feature_columns),
    "node_feature_columns": list(node_feature_columns),
    "label_columns": list(LABEL_COLUMNS),
}
write_json(feature_columns_payload, output_dir / "feature_columns_graph_cold_od_v2.json")

preprocessor_payload = {
    "edge_preprocessor": edge_preprocessor.to_dict(),
    "node_mean": node_mean.tolist(),
    "node_std": node_std.tolist(),
    "target_scale": float(target_scale),
    "year_to_idx": {str(k): int(v) for k, v in year_to_idx.items()},
}
write_json(preprocessor_payload, output_dir / "preprocessors_graph_cold_od_v2.json")


# ---------------------------------------------------------------------
# Save run configuration and artifact manifest.
# ---------------------------------------------------------------------

run_config = {
    "notebook": "GraphSAGE_HGT_ColdOD_Baselines_v2",
    "config": asdict(cfg),
    "paths": {
        "supervised_path": str(supervised_path),
        "manifest_path": str(manifest_path),
        "cold_baseline_predictions_path": str(cold_baseline_predictions_path),
        "cold_baseline_all_splits_path": str(cold_baseline_all_splits_path),
        "output_dir": str(output_dir),
    },
    "dataset": {
        "n_rows": int(len(df)),
        "split_counts": {str(k): int(v) for k, v in df["cold_split"].value_counts().to_dict().items()},
        "origin_column": str(origin_col),
        "destination_column": str(dest_col),
        "year_column": str(year_col),
        "n_nodes": int(len(zone_to_idx)),
        "edge_feature_dim": int(edge_attr_cpu.shape[1]),
        "node_feature_dim": int(node_x_cpu.shape[1]),
    },
    "package_versions": {
        "python": os.sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "torch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
    },
}
write_json(run_config, output_dir / "run_config_graph_cold_od_v2.json")

artifact_manifest = {
    "graph_by_seed_predictions": str(graph_by_seed_path),
    "graph_seed_ensemble_predictions": str(graph_seed_ensemble_path),
    "combined_predictions": str(combined_predictions_path),
    "leaderboard_test": str(leaderboard_path),
    "combined_metrics": str(combined_metrics_path),
    "seed_summary": str(seed_summary_path),
    "checkpoint_summary": str(checkpoint_summary_path),
    "training_history": str(history_path),
    "paired_summary_test_only": str(paired_summary_path),
    "paired_rows_test_only": paired_rows_saved_path,
    "tables_dir": str(tables_dir),
    "plots_dir": str(plots_dir),
    "models_dir": str(models_dir),
}
write_json(artifact_manifest, output_dir / "analysis_artifact_manifest_graph_cold_od_v2.json")


# ---------------------------------------------------------------------
# Save a lightweight dtype audit for debugging future Parquet issues.
# ---------------------------------------------------------------------

dtype_audit = pd.DataFrame(
    {
        "table": (
            ["graph_predictions_by_seed"] * len(graph_predictions_by_seed.columns)
            + ["graph_predictions_seed_ensemble"] * len(graph_predictions_seed_ensemble.columns)
            + ["combined_predictions"] * len(combined_predictions.columns)
        ),
        "column": (
            list(graph_predictions_by_seed.columns)
            + list(graph_predictions_seed_ensemble.columns)
            + list(combined_predictions.columns)
        ),
        "dtype_before_save": (
            [str(graph_predictions_by_seed[col].dtype) for col in graph_predictions_by_seed.columns]
            + [str(graph_predictions_seed_ensemble[col].dtype) for col in graph_predictions_seed_ensemble.columns]
            + [str(combined_predictions[col].dtype) for col in combined_predictions.columns]
        ),
    }
)
dtype_audit.to_csv(tables_dir / "parquet_dtype_audit_v2.csv", index=False)


print("Saved v2 artifacts to:", output_dir)
print(json.dumps(artifact_manifest, indent=2, default=json_default))

Saved v2 artifacts to: E:\NetworkOptimization\pythonProject1\Data\10_experiments\graphsage_hgt_cold_od_baselines_v2_notebook\east_plus_gulf
{
  "graph_by_seed_predictions": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\graphsage_hgt_cold_od_baselines_v2_notebook\\east_plus_gulf\\predictions_graph_cold_od_val_test_by_seed_v2.parquet",
  "graph_seed_ensemble_predictions": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\graphsage_hgt_cold_od_baselines_v2_notebook\\east_plus_gulf\\predictions_graph_cold_od_val_test_seed_ensemble_v2.parquet",
  "combined_predictions": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\graphsage_hgt_cold_od_baselines_v2_notebook\\east_plus_gulf\\combined_predictions_graph_cold_od_val_test_v2.parquet",
  "leaderboard_test": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\graphsage_hgt_cold_od_baselines_v2_notebook\\east_plus_gulf\\leaderboard_test_graph_cold_od_v2.csv",
  "combined_metrics": "E:\\

## 26. Generate a compact markdown report

The generated report summarizes the top seed-ensemble graph results, seed robustness, and strict test-only paired comparison against the strongest non-graph Cold-OD baseline.

In [28]:
def dataframe_to_markdown_safe(frame: pd.DataFrame, columns: Sequence[str], max_rows: int = 10) -> str:
    """Convert a dataframe slice to markdown without failing if optional columns are absent."""
    if frame.empty:
        return "_No rows available._"
    cols = [column for column in columns if column in frame.columns]
    return frame[cols].head(max_rows).to_markdown(index=False)


report_lines: List[str] = []
report_lines.append("# GraphSAGE/HGT Cold-OD Baselines v2 Report")
report_lines.append("")
report_lines.append("## Experiment summary")
report_lines.append(f"- Output directory: `{output_dir}`")
report_lines.append(f"- Seeds: `{list(cfg.seeds)}`")
report_lines.append(f"- Train/val/test row counts: `{df['cold_split'].value_counts().to_dict()}`")
report_lines.append("- Segment summaries: strict `split == test` only.")
report_lines.append("- Paired summaries: strict `split == test` only.")
report_lines.append("")

report_lines.append("## Top test leaderboard rows")
report_lines.append(dataframe_to_markdown_safe(
    leaderboard_test,
    ["rank", "source", "model", "checkpoint_metric", "seed", "pinball_mean", "weighted_pinball_mean", "mae_q75", "iqr_mae", "stress_top10_mae_q75"],
    max_rows=15,
))
report_lines.append("")

report_lines.append("## Seed robustness summary")
report_lines.append(dataframe_to_markdown_safe(
    seed_summary,
    ["source", "model", "checkpoint_metric", "n_seeds", "pinball_mean_mean", "pinball_mean_std", "mae_q75_mean", "mae_q75_std", "iqr_mae_mean", "iqr_mae_std"],
    max_rows=20,
))
report_lines.append("")

report_lines.append("## Test-only paired summary vs ColdOD::mlp_residual_prior_features")
report_lines.append(dataframe_to_markdown_safe(
    paired_summary_test_only,
    ["candidate_source", "candidate_model", "candidate_checkpoint_metric", "candidate_seed", "n_rows", "mean_delta_pinball_mean_row", "win_rate_pinball_mean_row", "mean_delta_abs_error_q75", "win_rate_abs_error_q75", "mean_delta_abs_error_iqr", "win_rate_abs_error_iqr"],
    max_rows=20,
))
report_lines.append("")

report_lines.append("## Interpretation notes")
report_lines.append("- A positive paired delta means the graph candidate is better than the non-graph reference on that test row.")
report_lines.append("- Compare `GRAPH_ENSEMBLE` rows first for the most stable multi-seed result.")
report_lines.append("- Use individual `GRAPH` seed rows to assess variance and robustness.")
report_lines.append("- If HGT remains stronger than GraphSAGE across seeds and checkpoints, typed relation modeling is supported.")
report_lines.append("- If GraphSAGE/HGT beat `ColdOD::mlp_residual_prior_features`, graph structure improves unseen-OD generalization.")

report_path = reports_dir / "graphsage_hgt_cold_od_v2_report.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print("Report written to:", report_path)
print("\n".join(report_lines[:30]))

ImportError: Missing optional dependency 'tabulate'.  Use pip or conda to install tabulate.